# Algorithm 10 CASAL-Chart-KLDM feasibility tests

This notebook implements the test tutorial from `/Users/glymov/Downloads/CASAL_testing(1) (1).md` as a unit-test-style feasibility suite.

It uses the same comparison config, subset seed, and sample seed as the current `mp20_sampling_compare_facit_pc_vs_algorithm10_casal_chart` run. Every test emits a dataframe with `PASS`, `status`, and debug fields so failures are visible rather than hidden. Oracle-Wyckoff tests and deployable non-oracle tests are separated so oracle feasibility cannot mask projection/search failures.

M1 CPU note: the setup cell auto-enables a CPU-heavy-but-bounded profile on Apple Silicon: the sampling-smoke cells run by default with 60 steps and K=1, the oracle SSVD identity test uses the full origin grid, deployable Algorithm10 projections keep a compact origin grid, and non-oracle template budgets are raised to 128/64. Set `KLDM_RUN_MODEL_SAMPLING_TESTS=0` when you only want the projection feasibility tests.


## 0. Setup and reproducibility

In [41]:
from __future__ import annotations

import dataclasses
import inspect
import itertools
import math
import os
import platform
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src" / "kldmPlus").exists():
    # Useful when the notebook is opened from a different working directory.
    for candidate in [Path("/workspace"), Path("/Users/glymov/DTU/6 Semester/Bachelor/Github/Main/kldm")]:
        if (candidate / "src" / "kldmPlus").exists():
            ROOT = candidate
            break
sys.path.insert(0, str(ROOT / "src"))

from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import evaluate_csp_reconstruction, aggregate_csp_reconstruction_metrics
from kldmPlus.symmetry.frame_bridge import standardize_structure, build_symmetry_frame_bridge
from pyxtal.symmetry import Group

from kldmPlus.symmetry.pyxtal_backend import build_pyxtal_wyckoff_result
from kldmPlus.symmetry.wyckoff_templates import (
    WyckoffTemplate,
    _lookup_wyckoff_position,
    _site_template_from_wp,
    composition_to_species_counts,
    extract_wyckoff_templates,
    flatten_site_signature,
    expand_wyckoff_template_torch,
    recover_template_free_vars_from_pyxtal_result,
    requested_conventional_atomic_numbers,
)
from kldmPlus.symmetry.k_basis import (
    cell_to_k,
    k_to_cell_matrix,
    k_to_free_vars,
    space_group_k_constraint,
)
from kldmPlus.symmetry.pcs_projection import (
    _build_vanilla_structure,
    _periodic_pairwise_distances,
    _species_assignment_indices,
    PCSTemplateState,
    initialize_constrained_template_states,
    materialize_pcs_state,
    validate_requested_space_group,
)
from kldmPlus.fixed_template_ssvd_project import (
    ProjectionMetric,
    SSVDProjectionConfig,
    fixed_template_ssvd_project,
)
from kldmPlus.algorithm10_casal_chart import (
    Algorithm10Config,
    _build_chart_target,
    _casal_step_graph,
    _decode_lattice_matrix,
    _init_graph_state,
    _k_to_l,
    _l_to_k,
    _materialize_projection,
    _origin_shift_candidates,
    _project_graph_to_chart,
)

CONFIG_PATH = ROOT / "configs" / "kldm_plus" / "mp_20" / "mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

COMPARE_CFG = CONFIG["sampling_compare"]
GRAPH_IDS = [1, 2, 3]  # 1-based, matching printed sample_debug idx values.
GRAPH_INDICES = [idx - 1 for idx in GRAPH_IDS]
SUBSET_SEED = int(COMPARE_CFG["subset_seed"])
SAMPLE_SEED = int(COMPARE_CFG["sample_seed"])

# Core gate tests run by default. Long sweeps and @20 reconstruction are explicit.
RUN_MODEL_SAMPLING_TESTS = True
RUN_SWEEP_K = False
RUN_RECONSTRUCTION_AT20 = False
MAX_SWEEP_COMBINATIONS = 8
K_RECON_HEAVY = 20
SAMPLING_SMOKE_N_STEPS = int(os.environ.get("KLDM_SAMPLING_SMOKE_N_STEPS", "60"))
SAMPLING_SMOKE_K = int(os.environ.get("KLDM_SAMPLING_SMOKE_K", "1"))
SAMPLING_SMOKE_PROJECTION_INTERVAL = int(os.environ.get("KLDM_SAMPLING_SMOKE_PROJECTION_INTERVAL", "10"))
SAMPLING_SMOKE_PROJECTION_START_FRACTION = float(os.environ.get("KLDM_SAMPLING_SMOKE_PROJECTION_START_FRACTION", "0.75"))
GRAPH3_K_RESIDUAL_WARN = float(os.environ.get("KLDM_GRAPH3_K_RESIDUAL_WARN", "0.25"))
CASAL_ABLATION_STEPS = int(os.environ.get("KLDM_CASAL_ABLATION_STEPS", "6"))
CASAL_ABLATION_IMPROVE_TOL = float(os.environ.get("KLDM_CASAL_ABLATION_IMPROVE_TOL", "1e-6"))
CASAL_ABLATION_ETA_SWEEP = tuple(
    float(x.strip())
    for x in os.environ.get("KLDM_CASAL_ABLATION_ETA_SWEEP", "0.01,0.025,0.05,0.1,0.25").split(",")
    if x.strip()
)
CASAL_ABLATION_MODES = tuple(
    x.strip()
    for x in os.environ.get(
        "KLDM_CASAL_ABLATION_MODES",
        "full_projection_dual,fixed_W_tau_pi_dual,coord_only_dual,lattice_only_dual",
    ).split(",")
    if x.strip()
)

# M1/CPU profile: keep pre-H tests runnable on a laptop CPU.
# Set KLDM_FORCE_M1_CPU_PROFILE=1 to force this profile on non-M1 CPU machines.
M1_CPU_PROFILE = False  # resolved after device is known
USE_EXACT_ORACLE_TEMPLATE = True
RUN_FULL_ORIGIN_GRID = True
ORACLE_TEMPLATE_MAX_TEMPLATES = None
ORACLE_TEMPLATE_EVAL_LIMIT = 4096
ORACLE_TEMPLATE_NMAX = 200000
CPU_WRONG_TEMPLATE_MAX = 8
CPU_WRONG_TEMPLATE_EVAL_LIMIT = 4

# Strict feasibility thresholds. These are intentionally stronger than "it ran" checks.
GT_PROJECTION_OBJECTIVE_TOL = 1.0e-4
GT_COORD_LOSS_TOL = 1.0e-5
GT_K_LOSS_TOL = 1.0e-5
SSVD_CONDITION_TOL = 1.0e6
WRONG_TEMPLATE_MARGIN = 1.0e-4
KLDM_PROJECTION_OBJECTIVE_TOL = 5.0
KLDM_MAX_PROJECTION_JUMP_FRAC = 0.40
KLDM_MAX_PROJECTION_JUMP_K = 2.50

np.random.seed(SAMPLE_SEED)
torch.manual_seed(SAMPLE_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SAMPLE_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
M1_CPU_PROFILE = bool(
    device.type == "cpu"
    and (platform.machine().lower() in {"arm64", "aarch64"} or os.environ.get("KLDM_FORCE_M1_CPU_PROFILE") == "1")
)
if M1_CPU_PROFILE:
    # Apple Silicon CPU profile: run the small sampling smoke test by default,
    # while keeping sweeps and @20 reconstruction opt-in.
    RUN_MODEL_SAMPLING_TESTS = os.environ.get("KLDM_RUN_MODEL_SAMPLING_TESTS", "1") != "0"
    RUN_SWEEP_K = os.environ.get("KLDM_RUN_SWEEP_K", "0") == "1"
    RUN_RECONSTRUCTION_AT20 = os.environ.get("KLDM_RUN_RECONSTRUCTION_AT20", "0") == "1"
    RUN_FULL_ORIGIN_GRID = os.environ.get("KLDM_RUN_FULL_ORIGIN_GRID", "1") != "0"
    torch.set_num_threads(max(1, min(6, os.cpu_count() or 6)))

origin_values = [0.0, 0.125, 0.875, 0.25, 0.75, 0.5]
origin_grid = list(itertools.product(origin_values, repeat=3))
compact_origin_grid = [
    (0.0, 0.0, 0.0),
    (0.125, 0.125, 0.125),
    (0.875, 0.875, 0.875),
    (0.25, 0.25, 0.25),
    (0.75, 0.75, 0.75),
    (0.5, 0.5, 0.5),
]
ssvd_origin_grid = origin_grid if RUN_FULL_ORIGIN_GRID else compact_origin_grid

print(f"ROOT={ROOT}")
print(f"CONFIG_PATH={CONFIG_PATH}")
print(f"subset_seed={SUBSET_SEED} sample_seed={SAMPLE_SEED} graph_ids={GRAPH_IDS}")
print(f"RUN_MODEL_SAMPLING_TESTS={RUN_MODEL_SAMPLING_TESTS} RUN_SWEEP_K={RUN_SWEEP_K} RUN_RECONSTRUCTION_AT20={RUN_RECONSTRUCTION_AT20}")
print(f"sampling_smoke n_steps={SAMPLING_SMOKE_N_STEPS} K={SAMPLING_SMOKE_K} projection_start_fraction={SAMPLING_SMOKE_PROJECTION_START_FRACTION} projection_interval={SAMPLING_SMOKE_PROJECTION_INTERVAL}")
print(f"casal_ablation steps={CASAL_ABLATION_STEPS} eta_sweep={CASAL_ABLATION_ETA_SWEEP} modes={CASAL_ABLATION_MODES}")
if RUN_MODEL_SAMPLING_TESTS:
    print("sampling_smoke_methods=KLDM, FinalProjection, Algorithm10-CASAL-lite")
print(f"device={device} machine={platform.machine()} M1_CPU_PROFILE={M1_CPU_PROFILE} RUN_FULL_ORIGIN_GRID={RUN_FULL_ORIGIN_GRID} ssvd_origin_count={len(ssvd_origin_grid)}")


ROOT=/workspace
CONFIG_PATH=/workspace/configs/kldm_plus/mp_20/mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml
subset_seed=1222 sample_seed=2026 graph_ids=[1, 2, 3]
RUN_MODEL_SAMPLING_TESTS=True RUN_SWEEP_K=False RUN_RECONSTRUCTION_AT20=False
sampling_smoke n_steps=60 K=1 projection_start_fraction=0.75 projection_interval=10
casal_ablation steps=6 eta_sweep=(0.01, 0.025, 0.05, 0.1, 0.25) modes=('full_projection_dual', 'fixed_W_tau_pi_dual', 'coord_only_dual', 'lattice_only_dual')
sampling_smoke_methods=KLDM, FinalProjection, Algorithm10-CASAL-lite
device=cpu machine=aarch64 M1_CPU_PROFILE=True RUN_FULL_ORIGIN_GRID=True ssvd_origin_count=216


## 1. Load trained KLDM and validation graphs

In [42]:

set_seed(SAMPLE_SEED)
runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
template_prior = runner._ensure_template_prior()
batch = next(iter(runner.loader)).to(runner.device)
ptr = batch.ptr.tolist()

sampler_cfgs = {cfg["name"]: dict(cfg) for cfg in COMPARE_CFG["samplers"]}
facit_cfg = sampler_cfgs.get("FacitKLDM_PC", COMPARE_CFG["samplers"][0])
algo10_cfg_map = dict(sampler_cfgs["Algorithm10_CASAL_chart"].get("algorithm10", {}))
BASE_ALGO10 = Algorithm10Config.from_mapping(algo10_cfg_map)

# Keep the notebook responsive while preserving the real algorithmic path.
# Set USE_FAST_DEBUG_CONFIG=False to use the exact config file budgets.
USE_FAST_DEBUG_CONFIG = True
if USE_FAST_DEBUG_CONFIG:
    # More expensive than the laptop smoke profile, but still bounded:
    # full origin-grid scans are used only by the explicit SSVD tests; deployable
    # Algorithm10 projections keep a compact origin grid.
    fast_max_templates = 128 if M1_CPU_PROFILE else 128
    fast_template_eval_limit = 64 if M1_CPU_PROFILE else 64
    fast_ssvd_steps = 16 if M1_CPU_PROFILE else 16
    fast_ssvd_restarts = 1 if M1_CPU_PROFILE else min(int(BASE_ALGO10.ssvd_random_restarts), 2)
    BASE_TEST_ALGO10 = dataclasses.replace(
        BASE_ALGO10,
        debug=False,
        max_templates=max(int(BASE_ALGO10.max_templates), fast_max_templates),
        template_eval_limit=max(int(BASE_ALGO10.template_eval_limit), fast_template_eval_limit),
        ssvd_random_restarts=fast_ssvd_restarts,
        ssvd_max_steps=max(int(BASE_ALGO10.ssvd_max_steps), fast_ssvd_steps),
        origin_shift_candidates=tuple(compact_origin_grid) if M1_CPU_PROFILE else tuple(BASE_ALGO10.origin_shift_candidates),
        debug_projection_examples=max(int(BASE_ALGO10.debug_projection_examples), 2),
    )
else:
    BASE_TEST_ALGO10 = dataclasses.replace(BASE_ALGO10, debug=False)

# Oracle config is only for ground-truth chart feasibility and diagnostics.
# Non-oracle config is the deployable path used by full projection/sampling tests.
ORACLE_ALGO10 = dataclasses.replace(
    BASE_TEST_ALGO10,
    oracle_wyckoff_debug=True,
    template_eval_limit=max(int(BASE_TEST_ALGO10.template_eval_limit), int(ORACLE_TEMPLATE_EVAL_LIMIT)),
    template_nmax=max(int(BASE_TEST_ALGO10.template_nmax), int(ORACLE_TEMPLATE_NMAX)),
)
NONORACLE_ALGO10 = dataclasses.replace(BASE_TEST_ALGO10, oracle_wyckoff_debug=False)
TEST_ALGO10 = NONORACLE_ALGO10

summary_rows = []
result_tables = {}
_caches = {}

print(f"batch.num_graphs={batch.num_graphs} batch.num_atoms={batch.num_atoms.tolist() if hasattr(batch.num_atoms, 'tolist') else batch.num_atoms}")
print(f"requested_sg={torch.as_tensor(batch.space_group).reshape(-1).tolist()}")
print(f"Algorithm10 profile={'fast_debug' if USE_FAST_DEBUG_CONFIG else 'exact_config'} max_templates={BASE_TEST_ALGO10.max_templates} template_eval_limit={BASE_TEST_ALGO10.template_eval_limit} ssvd_steps={BASE_TEST_ALGO10.ssvd_max_steps}")
print(f"oracle_wyckoff={ORACLE_ALGO10.oracle_wyckoff_debug} oracle_eval_limit={ORACLE_ALGO10.template_eval_limit} oracle_max_templates={ORACLE_TEMPLATE_MAX_TEMPLATES} nonoracle_wyckoff={NONORACLE_ALGO10.oracle_wyckoff_debug}")


mps:  False
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
dataset_cache action=load dataset=mp_20 split=train reason=fresh path=/workspace/data/mp_20/processed/train
dataset_cache action=from_cache_path:start dataset=mp_20 split=train
dataset_cache action=from_cache_path:done dataset=mp_20 split=train
dataset_cache action=builder_build:start dataset=mp_20 split=train
dataset_cache action=builder_build:done dataset=mp_20 split=train
template_prior_cache_hit path=/workspace/notebooks/.cache/kldmPlus/template_prior/0448db66e575a7bb.pkl records=43
batch.num_graphs=3 batch.num_atoms=[6, 16, 8]
requested_sg=[227, 4, 194]
Algorithm10 profile=fast_debug max_templates=128 te

## 2. Shared helper metrics and fixtures

In [43]:
def wrap_delta(x):
    return torch.remainder(x + 0.5, 1.0) - 0.5


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def graph_tensors(graph_idx0: int, *, source=None):
    if source is None:
        start, end = graph_slice(graph_idx0)
        return {
            "pos": batch.pos[start:end].detach().clone(),
            "l": batch.l[graph_idx0].detach().clone().reshape(-1),
            "h": batch.atomic_numbers[start:end].detach().clone().to(torch.long),
            "sg": int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
            "num_atoms": int(end - start),
        }
    if len(source) == 4:
        pos, _v, l, h = source
    else:
        pos, l, h = source
    start, end = graph_slice(graph_idx0)
    return {
        "pos": pos[start:end].detach().clone(),
        "l": l[graph_idx0].detach().clone().reshape(-1),
        "h": h[start:end].detach().clone().to(torch.long),
        "sg": int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
        "num_atoms": int(end - start),
    }


def cell_from_l(l, num_atoms):
    return _decode_lattice_matrix(l=l.reshape(-1), num_atoms=int(num_atoms), lattice_transform=runner.lattice_transform).detach()


def structure_from_graph(graph_idx0: int, *, source=None):
    g = graph_tensors(graph_idx0, source=source)
    return _build_vanilla_structure(
        frac_coords=g["pos"],
        atomic_numbers=g["h"],
        cell_matrix=cell_from_l(g["l"], g["num_atoms"]),
    )


def detect_sg(structure, symprec=1e-2, angle_tolerance=5.0):
    try:
        return int(SpacegroupAnalyzer(structure, symprec=symprec, angle_tolerance=angle_tolerance).get_space_group_number())
    except Exception:
        return None


def min_pair_distance_structure(structure):
    D = np.asarray(structure.distance_matrix, dtype=float)
    if D.shape[0] < 2:
        return float("inf")
    D = D + np.diag(np.full(D.shape[0], 1e9))
    return float(np.min(D))


def composition_counter(values):
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    return dict(sorted(Counter(arr).items()))


def species_matched_torus_metrics(source_frac, source_z, target_frac, target_z):
    assignment = _species_assignment_indices(
        source_frac=source_frac,
        source_atomic_numbers=source_z.to(device=source_frac.device, dtype=torch.long),
        target_frac=target_frac,
        target_atomic_numbers=target_z.to(device=target_frac.device, dtype=torch.long),
    ).to(device=target_frac.device, dtype=torch.long)
    matched = target_frac[assignment]
    delta = wrap_delta(source_frac - matched)
    loss = float(delta.square().mean().detach().item()) if delta.numel() else 0.0
    rmse = float(torch.sqrt(delta.square().mean()).detach().item()) if delta.numel() else 0.0
    distances = torch.linalg.norm(delta, dim=-1)
    return {
        "assignment": assignment,
        "assignment_cost": loss,
        "frac_rmse": rmse,
        "mean_frac_dist": float(distances.mean().detach().item()) if distances.numel() else 0.0,
        "max_frac_dist": float(distances.max().detach().item()) if distances.numel() else 0.0,
    }


def best_tau_metrics(source_frac, source_z, target_frac, target_z, grid=origin_grid):
    best = None
    for tau_values in grid:
        tau = torch.tensor(tau_values, device=source_frac.device, dtype=source_frac.dtype).reshape(1, 3)
        metrics = species_matched_torus_metrics(torch.remainder(source_frac + tau, 1.0), source_z, target_frac, target_z)
        row = {"best_tau": tuple(float(v) for v in tau_values), **metrics}
        if best is None or row["assignment_cost"] < best["assignment_cost"]:
            best = row
    return best


def tensor_cell_lengths_angles(cell):
    lengths = torch.linalg.norm(cell, dim=-1)
    a, b, c = [cell[i] for i in range(3)]
    angles = torch.stack([
        torch.acos(torch.clamp(torch.dot(b, c) / (torch.linalg.norm(b) * torch.linalg.norm(c)).clamp_min(1e-8), -1.0, 1.0)),
        torch.acos(torch.clamp(torch.dot(a, c) / (torch.linalg.norm(a) * torch.linalg.norm(c)).clamp_min(1e-8), -1.0, 1.0)),
        torch.acos(torch.clamp(torch.dot(a, b) / (torch.linalg.norm(a) * torch.linalg.norm(b)).clamp_min(1e-8), -1.0, 1.0)),
    ]) * 180.0 / math.pi
    return lengths, angles


def family_name(sg: int):
    if 195 <= sg <= 230:
        return "cubic"
    if 168 <= sg <= 194:
        return "hexagonal_trigonal"
    if 143 <= sg <= 167:
        return "trigonal"
    if 75 <= sg <= 142:
        return "tetragonal"
    if 16 <= sg <= 74:
        return "orthorhombic"
    if 3 <= sg <= 15:
        return "monoclinic"
    return "triclinic"


def family_constraints_ok(sg, cell, tol_len=1e-3, tol_angle=1e-2):
    lengths, angles = tensor_cell_lengths_angles(cell)
    L = lengths.detach().cpu().numpy()
    A = angles.detach().cpu().numpy()
    fam = family_name(int(sg))
    if fam == "cubic":
        return bool(np.max(np.abs(L - L.mean())) < tol_len and np.max(np.abs(A - 90.0)) < tol_angle)
    if fam == "tetragonal":
        return bool(abs(L[0] - L[1]) < tol_len and np.max(np.abs(A - 90.0)) < tol_angle)
    if fam == "hexagonal_trigonal":
        return bool(abs(L[0] - L[1]) < tol_len and min(abs(A[2] - 120.0), abs(A[2] - 60.0)) < 1e-1)
    if fam == "orthorhombic":
        return bool(np.max(np.abs(A - 90.0)) < tol_angle)
    if fam == "monoclinic":
        return bool(abs(A[0] - 90.0) < 1e-1 and abs(A[2] - 90.0) < 1e-1)
    return bool(torch.det(cell).abs().item() > 0)


def dataframe_result(name, rows):
    df = pd.DataFrame(rows)
    if "PASS" not in df.columns:
        df["PASS"] = False
    if "status" not in df.columns:
        df["status"] = np.where(df["PASS"].astype(bool), "PASS", "FAIL")
    result_tables[name] = df
    status_series = df.get("status", pd.Series(["PASS" if bool(v) else "FAIL" for v in df["PASS"].fillna(False)], dtype=str)).astype(str)
    skip_count = int((status_series == "SKIP").sum())
    error_count = int((status_series == "ERROR").sum())
    fail_mask = (~df["PASS"].fillna(False).astype(bool)) & (status_series != "SKIP") & (status_series != "ERROR")
    failish = df[fail_mask | (status_series == "ERROR")]
    summary_rows.append({
        "test_name": name,
        "graphs_tested": int(df["graph"].nunique()) if "graph" in df.columns else int(len(df)),
        "pass_count": int(df["PASS"].fillna(False).sum()),
        "fail_count": int(fail_mask.sum()),
        "skip_count": skip_count,
        "error_count": error_count,
        "worst_graph": None if failish.empty or "graph" not in failish.columns else failish.iloc[0]["graph"],
        "failure_category": None if failish.empty else failish.iloc[0].get("failure_category", "UNKNOWN"),
        "action_needed": None if failish.empty else failish.iloc[0].get("action_needed", "inspect debug columns"),
    })
    return df


def error_row(test_name, graph, exc, category="ERROR", action="inspect exception"):
    return {
        "test": test_name,
        "graph": graph,
        "PASS": False,
        "status": "ERROR",
        "error": f"{type(exc).__name__}: {exc}",
        "failure_category": category,
        "action_needed": action,
    }


def skipped_df(name, reason, graphs=GRAPH_IDS):
    rows = [{"test": name, "graph": g, "PASS": False, "status": "SKIP", "reason": reason, "failure_category": "SKIPPED_BY_FLAG", "action_needed": "enable the corresponding RUN_* flag"} for g in graphs]
    return dataframe_result(name, rows)


def standard_graph_structure(graph_idx0):
    vanilla = structure_from_graph(graph_idx0)
    analyzer, standardized = standardize_structure(vanilla, standardization="conventional", symprec=TEST_ALGO10.symprec, angle_tolerance=TEST_ALGO10.angle_tolerance)
    return vanilla, standardized


def pyxtal_result_for_graph(graph_idx0):
    key = ("pyxtal", graph_idx0)
    if key not in _caches:
        _, standardized = standard_graph_structure(graph_idx0)
        _caches[key] = build_pyxtal_wyckoff_result(standardized, symprec=TEST_ALGO10.symprec, pyxtal_tol=TEST_ALGO10.symprec)
    return _caches[key]


def oracle_signature_for_graph(graph_idx0):
    result = pyxtal_result_for_graph(graph_idx0)
    return tuple(sorted([(int(z), str(label)) for z, label in zip(result.anchor_atomic_numbers.tolist(), result.site_labels)], key=lambda item: (item[0], item[1])))




def exact_oracle_template_for_graph(graph_idx0):
    """Build the exact oracle Wyckoff template from PyXtal extraction without brute-force enumeration."""
    key = ("exact_oracle_template", graph_idx0)
    if key in _caches:
        return _caches[key]
    result = pyxtal_result_for_graph(graph_idx0)
    group = Group(int(result.space_group))
    site_templates = []
    for atomic_number, label in zip(result.anchor_atomic_numbers.tolist(), result.site_labels):
        wp = _lookup_wyckoff_position(group, str(label))
        site_templates.append(_site_template_from_wp(atomic_number=int(atomic_number), wp=wp))
    species_order, species_counts = composition_to_species_counts(torch.as_tensor(result.expanded_atomic_numbers, dtype=torch.long))
    template = WyckoffTemplate(
        space_group=int(result.space_group),
        group_symbol=str(group.symbol),
        species_order=tuple(int(v) for v in species_order),
        species_counts=tuple(int(v) for v in species_counts),
        site_templates=tuple(site_templates),
        has_free_coordinates=bool(any(site.dof > 0 for site in site_templates)),
        pyxtal_site_index_groups=None,
    )
    _caches[key] = template
    return template


def exact_oracle_template_state_for_graph(graph_idx0):
    key = ("exact_oracle_state", graph_idx0)
    if key in _caches:
        return _caches[key]
    g = graph_tensors(graph_idx0)
    vanilla = structure_from_graph(graph_idx0)
    bridge = build_symmetry_frame_bridge(
        vanilla_structure=vanilla,
        standardization="conventional",
        symprec=ORACLE_ALGO10.symprec,
        angle_tolerance=ORACLE_ALGO10.angle_tolerance,
    )
    cell = cell_from_l(g["l"], g["num_atoms"]).to(device=runner.device, dtype=dtype)
    chart_frac, chart_h, chart_cell, chart_k = _build_chart_target(
        frac_coords=g["pos"].to(device=runner.device, dtype=dtype),
        atomic_numbers=g["h"].to(device=runner.device, dtype=torch.long),
        cell_matrix=cell,
        requested_sg=g["sg"],
        symprec=ORACLE_ALGO10.symprec,
        angle_tolerance=ORACLE_ALGO10.angle_tolerance,
    )
    constraint = space_group_k_constraint(space_group_number=g["sg"], device=runner.device, dtype=dtype)
    lattice_free_vars = k_to_free_vars(chart_k, constraint).detach().clone()
    template = exact_oracle_template_for_graph(graph_idx0)
    free_vars = recover_template_free_vars_from_pyxtal_result(template, pyxtal_result_for_graph(graph_idx0)).to(device=runner.device, dtype=dtype)
    state = PCSTemplateState(
        template=template,
        constraint=constraint,
        bridge=bridge,
        free_vars=free_vars.detach().clone(),
        lattice_free_vars=lattice_free_vars.detach().clone(),
        objective=0.0,
        template_rank=1,
        candidate_count=1,
        ranking_objective=0.0,
        prior_score=0,
        target_frac=chart_frac.detach().clone(),
        target_atomic_numbers=chart_h.detach().clone(),
        target_cell=chart_cell.detach().clone(),
        target_k=chart_k.detach().clone(),
        anchor_cell=chart_cell.detach().clone(),
        anchor_k=chart_k.detach().clone(),
        anchor_lattice_free_vars=lattice_free_vars.detach().clone(),
        target_representation_name="exact_oracle_chart_target",
    )
    _caches[key] = state
    return state

def templates_for_graph(graph_idx0, *, max_templates="oracle", config=None):
    cfg = config or ORACLE_ALGO10
    resolved_max = ORACLE_TEMPLATE_MAX_TEMPLATES if max_templates == "oracle" else max_templates
    key = ("templates", graph_idx0, resolved_max, int(cfg.template_nmax), bool(cfg.quick_templates))
    if key not in _caches:
        g = graph_tensors(graph_idx0)
        requested_atomic_numbers = requested_conventional_atomic_numbers(g["h"], space_group_number=g["sg"])
        _caches[key] = extract_wyckoff_templates(
            space_group_number=g["sg"],
            atomic_numbers=requested_atomic_numbers,
            max_templates=resolved_max,
            quick=bool(cfg.quick_templates),
            nmax=int(cfg.template_nmax),
        )
    return _caches[key]


def oracle_template_for_graph(graph_idx0):
    if USE_EXACT_ORACLE_TEMPLATE:
        return exact_oracle_template_for_graph(graph_idx0)
    oracle_sig = oracle_signature_for_graph(graph_idx0)
    candidates = templates_for_graph(graph_idx0, max_templates="oracle", config=ORACLE_ALGO10)
    for template in candidates:
        if flatten_site_signature(template) == oracle_sig:
            return template
    return None


def ssvd_state_for_graph(graph_idx0, *, require_oracle=True):
    if USE_EXACT_ORACLE_TEMPLATE:
        return exact_oracle_template_state_for_graph(graph_idx0)
    g = graph_tensors(graph_idx0)
    target_cell = cell_from_l(g["l"], g["num_atoms"]).to(device=g["pos"].device, dtype=g["pos"].dtype)
    states = initialize_constrained_template_states(
        reference_frac_coords=g["pos"],
        atomic_numbers=g["h"],
        cell_matrix=target_cell,
        space_group_number=g["sg"],
        standardization="conventional",
        symprec=ORACLE_ALGO10.symprec,
        angle_tolerance=ORACLE_ALGO10.angle_tolerance,
        max_templates=ORACLE_TEMPLATE_MAX_TEMPLATES,
        template_eval_limit=ORACLE_ALGO10.template_eval_limit,
        quick_templates=ORACLE_ALGO10.quick_templates,
        top_k=max(1, ORACLE_ALGO10.template_eval_limit),
        template_prior=template_prior,
        template_prior_weight=ORACLE_ALGO10.template_prior_weight,
        debug_template_candidates=False,
    )
    oracle_sig = oracle_signature_for_graph(graph_idx0)
    matches = [state for state in states if flatten_site_signature(state.template) == oracle_sig]
    if require_oracle and not matches:
        available = [flatten_site_signature(state.template) for state in states[:10]]
        raise RuntimeError(f"oracle template signature not found in initialized states: {oracle_sig}; first_available={available}; states={len(states)}")
    return matches[0] if matches else states[0]


def facit_smoke_cfg():
    cfg = dict(facit_cfg)
    cfg["n_steps"] = int(SAMPLING_SMOKE_N_STEPS)
    return cfg


def algorithm10_smoke_cfg(*, updates=None):
    smoke_updates = {
        "projection_start_fraction": float(SAMPLING_SMOKE_PROJECTION_START_FRACTION),
        "projection_interval": int(SAMPLING_SMOKE_PROJECTION_INTERVAL),
    }
    if updates:
        smoke_updates.update(dict(updates))
    return sampler_cfg_with_algorithm10(
        sampler_cfgs["Algorithm10_CASAL_chart"],
        NONORACLE_ALGO10,
        n_steps=int(SAMPLING_SMOKE_N_STEPS),
        updates=smoke_updates,
    )


def get_kldm_sample_once():
    key = ("kldm_sample", int(SAMPLING_SMOKE_N_STEPS))
    if key not in _caches:
        if not RUN_MODEL_SAMPLING_TESTS:
            raise RuntimeError("RUN_MODEL_SAMPLING_TESTS=False")
        set_seed(SAMPLE_SEED)
        pos, _v, l, h = runner._sample_batch(batch, sampler_cfg=facit_smoke_cfg())
        _caches[key] = (pos, l, h)
    return _caches[key]


def evaluate_prediction_rows(name, pos, l, h, *, method):
    rows = []
    for graph_idx0, graph_id in zip(GRAPH_INDICES, GRAPH_IDS):
        start, end = graph_slice(graph_idx0)
        target = graph_tensors(graph_idx0)
        result = evaluate_csp_reconstruction(
            pred_f=pos[start:end],
            pred_l=l[graph_idx0],
            pred_a=h[start:end],
            target_f=target["pos"],
            target_l=target["l"],
            target_a=target["h"],
            lattice_transform=runner.lattice_transform,
            requested_space_group=target["sg"],
            validity_cutoff=float(COMPARE_CFG.get("validity_cutoff", 1.0)),
        )
        rows.append({
            "test": name,
            "method": method,
            "graph": graph_id,
            "valid": bool(result.valid),
            "composition_match": result.composition_match,
            "requested_sg_match": result.requested_space_group_match,
            "detected_sg": result.detected_space_group,
            "structure_match": bool(result.match),
            "rmse": result.rmse,
            "min_pair_distance": result.min_pair_distance,
            "volume": result.volume,
            "validity_reason": result.validity_reason,
            "PASS": bool(result.valid and result.composition_match and (result.requested_space_group_match is not False)),
            "status": "PASS" if bool(result.valid) else "FAIL",
        })
    return rows

# Override/elevate the evaluation helpers with method_base/k parsing and single-graph support.
def method_base_from_label(method: str) -> str:
    return str(method).split("@", 1)[0]


def method_k_from_label(method: str) -> int:
    text = str(method)
    if "@" not in text:
        return 1
    tail = text.split("@", 1)[1]
    try:
        return int(tail.split("#", 1)[0])
    except Exception:
        return 1


def algo10_config_payload(config: Algorithm10Config) -> dict:
    payload = dataclasses.asdict(config)
    payload["origin_shift_values"] = list(payload.get("origin_shift_values", []))
    payload["origin_shift_candidates"] = [list(v) for v in payload.get("origin_shift_candidates", [])]
    payload["ssvd_line_search_alphas"] = list(payload.get("ssvd_line_search_alphas", []))
    return payload


def sampler_cfg_with_algorithm10(base_cfg: dict, config: Algorithm10Config, *, n_steps=None, updates=None) -> dict:
    cfg = dict(base_cfg)
    if n_steps is not None:
        cfg["n_steps"] = int(n_steps)
    cfg["algorithm10"] = algo10_config_payload(config)
    if updates:
        cfg["algorithm10"].update(dict(updates))
    cfg["algorithm10"]["debug"] = False
    return cfg


def oracle_reference_for_config(graph_idx0: int, config: Algorithm10Config):
    return structure_from_graph(graph_idx0) if bool(config.oracle_wyckoff_debug) else None


def target_volume_for_graph(graph_idx0: int, *, source=None) -> float:
    return float(structure_from_graph(graph_idx0, source=source).volume)


def projection_volume_ratio(graph_idx0: int, projection, *, source=None) -> float:
    cell = cell_from_l(projection.l, int(projection.pos.shape[0]))
    volume = float(torch.abs(torch.linalg.det(cell)).detach().item())
    return volume / max(target_volume_for_graph(graph_idx0, source=source), 1.0e-8)


def volume_ratio_ok(config: Algorithm10Config, ratio: float) -> bool:
    return bool(float(config.hard_volume_ratio_min) <= float(ratio) <= float(config.hard_volume_ratio_max))


def structure_from_fixed_projection(result):
    return _build_vanilla_structure(
        frac_coords=result.frac_coords_chart,
        atomic_numbers=result.atomic_numbers_chart,
        cell_matrix=result.cell,
    )


def fixed_template_best_over_origin_grid(template_state, *, y_f, y_k, y_h, config: Algorithm10Config):
    metric = config.projection_metric()
    ssvd_cfg = config.ssvd_projection_config()
    tau_candidates = [torch.tensor(shift, device=y_f.device, dtype=y_f.dtype).reshape(1, 3) for shift in ssvd_origin_grid]
    best = None
    results = []
    errors = []
    for tau0 in tau_candidates:
        try:
            result = fixed_template_ssvd_project(
                template_state=template_state,
                y_f=y_f,
                y_k=y_k,
                y_h=y_h.to(device=y_f.device, dtype=torch.long),
                tau0=tau0,
                metric=metric,
                config=ssvd_cfg,
            )
            results.append(result)
            if best is None or float(result.objective) < float(best.objective):
                best = result
        except Exception as exc:
            errors.append(f"tau={tuple(round(float(v), 4) for v in tau0.reshape(-1).detach().cpu().tolist())}: {type(exc).__name__}: {exc}")
    if best is None:
        raise RuntimeError("all origin-grid SSVD projections failed: " + " | ".join(errors[:4]))
    return best, results, errors


def evaluate_graph_prediction_row(name, graph_idx0, *, pred_pos, pred_l, pred_h, method):
    target = graph_tensors(graph_idx0)
    result = evaluate_csp_reconstruction(
        pred_f=pred_pos,
        pred_l=pred_l.reshape(-1),
        pred_a=pred_h.to(dtype=torch.long),
        target_f=target["pos"],
        target_l=target["l"],
        target_a=target["h"],
        lattice_transform=runner.lattice_transform,
        requested_space_group=target["sg"],
        validity_cutoff=float(COMPARE_CFG.get("validity_cutoff", 1.0)),
    )
    return {
        "test": name,
        "method": method,
        "method_base": method_base_from_label(method),
        "k": method_k_from_label(method),
        "graph": GRAPH_IDS[GRAPH_INDICES.index(graph_idx0)],
        "valid": bool(result.valid),
        "composition_match": result.composition_match,
        "requested_sg_match": result.requested_space_group_match,
        "detected_sg": result.detected_space_group,
        "structure_match": bool(result.match),
        "rmse": result.rmse,
        "min_pair_distance": result.min_pair_distance,
        "volume": result.volume,
        "validity_reason": result.validity_reason,
        "PASS": bool(result.valid and result.composition_match and (result.requested_space_group_match is not False)),
        "status": "PASS" if bool(result.valid and result.composition_match) else "FAIL",
    }


def evaluate_prediction_rows(name, pos, l, h, *, method):
    rows = []
    for graph_idx0 in GRAPH_INDICES:
        start, end = graph_slice(graph_idx0)
        rows.append(evaluate_graph_prediction_row(
            name,
            graph_idx0,
            pred_pos=pos[start:end],
            pred_l=l[graph_idx0],
            pred_h=h[start:end],
            method=method,
        ))
    return rows


def project_source_rows_to_chart(name, source, *, config: Algorithm10Config, method: str):
    rows = []
    pos_parts = []
    l_parts = []
    h_parts = []
    projection_success = True
    for graph_idx0 in GRAPH_INDICES:
        graph_id = GRAPH_IDS[GRAPH_INDICES.index(graph_idx0)]
        try:
            g = graph_tensors(graph_idx0, source=source)
            projection = _project_graph_to_chart(
                graph_idx=graph_idx0,
                requested_sg=g["sg"],
                target_pos=g["pos"],
                target_l=g["l"],
                target_h=g["h"],
                lattice_transform=runner.lattice_transform,
                config=config,
                template_prior=template_prior,
                oracle_reference_structure=oracle_reference_for_config(graph_idx0, config),
            )
            pos_parts.append(projection.pos.detach().clone())
            l_parts.append(projection.l.detach().clone().reshape_as(batch.l[graph_idx0]))
            h_parts.append(projection.h.detach().clone())
            row = evaluate_graph_prediction_row(
                name,
                graph_idx0,
                pred_pos=projection.pos,
                pred_l=projection.l,
                pred_h=projection.h,
                method=method,
            )
            row.update({
                "projection_objective": projection.objective,
                "projection_coord_loss": projection.coord_loss,
                "projection_lattice_k_loss": projection.lattice_k_loss,
                "projection_min_pair": projection.min_pair_distance,
                "projection_volume_ratio": projection_volume_ratio(graph_idx0, projection, source=source),
                "projection_template_labels": projection.template_labels,
                "projection_tau": tuple(round(float(v), 4) for v in projection.tau.reshape(-1).detach().cpu().tolist()),
            })
            rows.append(row)
        except Exception as exc:
            projection_success = False
            row = error_row(name, graph_id, exc, "FINAL_PROJECTION_ERROR", "inspect non-oracle chart projection from KLDM sample")
            row.update({"method": method, "method_base": method_base_from_label(method), "k": method_k_from_label(method)})
            rows.append(row)
    if projection_success and pos_parts:
        _caches[("method_source", method)] = (torch.cat(pos_parts, dim=0), torch.stack(l_parts, dim=0), torch.cat(h_parts, dim=0))
    return rows


## 3. Test A — data extraction and graph identity

**Actually tests:** Loads exactly the same seeded validation graphs as the compare run and checks composition, requested SG labels, lattice sanity, and target evaluability.


In [44]:
rows = []
for graph_idx0, graph_id in zip(GRAPH_INDICES, GRAPH_IDS):
    try:
        g = graph_tensors(graph_idx0)
        vanilla, standardized = standard_graph_structure(graph_idx0)
        detected_vanilla = detect_sg(vanilla, TEST_ALGO10.symprec, TEST_ALGO10.angle_tolerance)
        detected_standardized = detect_sg(standardized, TEST_ALGO10.symprec, TEST_ALGO10.angle_tolerance)
        passed = detected_standardized == g["sg"]
        rows.append({
            "test": "A_data_extraction",
            "graph": graph_id,
            "num_atoms": g["num_atoms"],
            "requested_sg": g["sg"],
            "detected_sg_vanilla": detected_vanilla,
            "detected_sg_standardized": detected_standardized,
            "volume_vanilla": float(vanilla.volume),
            "volume_standardized": float(standardized.volume),
            "composition": vanilla.composition.formula,
            "PASS": passed,
            "status": "PASS" if passed else "FAIL",
            "failure_category": None if passed else "FRAME_BRIDGE_FAILURE",
            "action_needed": None if passed else "debug standardization, symprec, and requested SG labels",
        })
    except Exception as exc:
        rows.append(error_row("A_data_extraction", graph_id, exc, "FRAME_BRIDGE_FAILURE"))
df_A = dataframe_result("A_data_extraction", rows)
df_A


,test,graph,num_atoms,requested_sg,detected_sg_vanilla,detected_sg_standardized,volume_vanilla,volume_standardized,composition,PASS,status,failure_category,action_needed
0,A_data_extraction,1,6,227,227,227,107.026724,428.106896,Yb2 Ir4,True,PASS,None,None
1,A_data_extraction,2,16,4,4,4,154.887316,154.887316,Fe6 Co6 B4,True,PASS,None,None
2,A_data_extraction,3,8,194,194,194,149.449362,149.449347,Hf2 Ti6,True,PASS,None,None


## 4. Test B — PyXtal / Wyckoff extraction sanity

**Actually tests:** Checks that ground-truth structures can be standardized and decomposed into PyXtal Wyckoff sites without losing composition or requested symmetry.


In [45]:
rows = []
for graph_idx0, graph_id in zip(GRAPH_INDICES, GRAPH_IDS):
    try:
        _, standardized = standard_graph_structure(graph_idx0)
        result = pyxtal_result_for_graph(graph_idx0)
        expected_num_atoms = int(len(standardized))
        expanded_counts = composition_counter(result.expanded_atomic_numbers)
        target_counts = composition_counter(standardized.atomic_numbers)
        multiplicity_sum = int(np.sum(result.site_multiplicities))
        passed = (
            int(result.num_atoms) == expected_num_atoms
            and multiplicity_sum == expected_num_atoms
            and expanded_counts == target_counts
        )
        rows.append({
            "test": "B_pyxtal_extraction",
            "graph": graph_id,
            "sg": int(result.space_group),
            "site_labels": list(result.site_labels),
            "site_multiplicities": result.site_multiplicities.tolist(),
            "site_dofs": result.site_dofs.tolist(),
            "anchor_count": int(result.anchor_count),
            "expanded_num_atoms": int(result.num_atoms),
            "expected_num_atoms": expected_num_atoms,
            "expanded_counts": expanded_counts,
            "target_counts": target_counts,
            "PASS": passed,
            "status": "PASS" if passed else "FAIL",
            "failure_category": None if passed else "PYXTAL_EXTRACTION_FAILURE",
            "action_needed": None if passed else "sweep symprec/pyxtal_tol or inspect primitive/conventional frame",
        })
    except Exception as exc:
        rows.append(error_row("B_pyxtal_extraction", graph_id, exc, "PYXTAL_EXTRACTION_FAILURE"))
df_B = dataframe_result("B_pyxtal_extraction", rows)
df_B


,test,graph,sg,site_labels,site_multiplicities,site_dofs,anchor_count,expanded_num_atoms,expected_num_atoms,expanded_counts,target_counts,PASS,status,failure_category,action_needed
0,B_pyxtal_extraction,1,227,"[8a, 16d]","[8, 16]","[0, 0]",2,24,24,"{70: 8, 77: 16}","{70: 8, 77: 16}",True,PASS,None,None
1,B_pyxtal_extraction,2,4,"[2a, 2a, 2a, 2a, 2a, 2a, 2a, 2a]","[2, 2, 2, 2, 2, 2, 2, 2]","[3, 3, 3, 3, 3, 3, 3, 3]",8,16,16,"{5: 4, 26: 6, 27: 6}","{5: 4, 26: 6, 27: 6}",True,PASS,None,None
2,B_pyxtal_extraction,3,194,"[2d, 6h]","[2, 6]","[0, 1]",2,8,8,"{22: 6, 72: 2}","{22: 6, 72: 2}",True,PASS,None,None


## 5. Test C — DiffCSP++ template enumeration sanity

**Actually tests:** Checks whether the template enumerator can recover the oracle Wyckoff signature for each target graph.


In [46]:

rows = []
for graph_idx0, graph_id in zip(GRAPH_INDICES, GRAPH_IDS):
    try:
        g = graph_tensors(graph_idx0)
        oracle_sig = oracle_signature_for_graph(graph_idx0)
        exact_template = exact_oracle_template_for_graph(graph_idx0) if USE_EXACT_ORACLE_TEMPLATE else None
        exact_sig = flatten_site_signature(exact_template) if exact_template is not None else None
        shallow_templates = templates_for_graph(graph_idx0, max_templates=BASE_TEST_ALGO10.max_templates, config=BASE_TEST_ALGO10)
        shallow_sigs = [flatten_site_signature(t) for t in shallow_templates]
        shallow_found = oracle_sig in shallow_sigs
        if USE_EXACT_ORACLE_TEMPLATE:
            found = exact_sig == oracle_sig
            rank = None
            num_templates = len(shallow_templates)
            source = "exact_pyxtal_template"
        else:
            templates = templates_for_graph(graph_idx0, max_templates="oracle", config=ORACLE_ALGO10)
            candidate_sigs = [flatten_site_signature(t) for t in templates]
            found = oracle_sig in candidate_sigs
            rank = candidate_sigs.index(oracle_sig) + 1 if found else None
            num_templates = len(templates)
            source = "deep_enumeration"
        rows.append({
            "test": "C_template_enumeration",
            "graph": graph_id,
            "sg": g["sg"],
            "oracle_template_source": source,
            "num_templates_checked": num_templates,
            "num_shallow_templates": len(shallow_templates),
            "oracle_signature": oracle_sig,
            "oracle_signature_found": found,
            "oracle_signature_found_in_shallow_pool": shallow_found,
            "oracle_rank_if_deep_enumerated": rank,
            "PASS": bool(found),
            "status": "PASS" if found else "FAIL",
            "failure_category": None if found else "TEMPLATE_NOT_AVAILABLE",
            "action_needed": None if found else "exact PyXtal oracle template could not be built; inspect labels/species/order",
        })
    except Exception as exc:
        rows.append(error_row("C_template_enumeration", graph_id, exc, "TEMPLATE_NOT_AVAILABLE"))
df_C = dataframe_result("C_template_enumeration", rows)
df_C


,test,graph,sg,oracle_template_source,num_templates_checked,num_shallow_templates,oracle_signature,oracle_signature_found,oracle_signature_found_in_shallow_pool,oracle_rank_if_deep_enumerated,PASS,status,failure_category,action_needed
0,C_template_enumeration,1,227,exact_pyxtal_template,4,4,"((70, 8a), (77, 16d))",True,True,None,True,PASS,None,None
1,C_template_enumeration,2,4,exact_pyxtal_template,1,1,"((5, 2a), (5, 2a), (26, 2a), (26, 2a), (26, 2a...",True,True,None,True,PASS,None,None
2,C_template_enumeration,3,194,exact_pyxtal_template,36,36,"((22, 6h), (72, 2d))",True,True,None,True,PASS,None,None


## 6. Test D — materialization identity and origin shift

**Actually tests:** Checks that oracle templates materialize back to the ground-truth motif, including origin-shift recovery.


In [47]:
rows = []
for graph_idx0, graph_id in zip(GRAPH_INDICES, GRAPH_IDS):
    try:
        _, standardized = standard_graph_structure(graph_idx0)
        target_frac = torch.as_tensor(np.array(standardized.frac_coords, dtype=float), device=runner.device, dtype=dtype)
        target_z = torch.as_tensor(np.array(standardized.atomic_numbers, dtype=int), device=runner.device, dtype=torch.long)
        template = oracle_template_for_graph(graph_idx0)
        if template is None:
            raise RuntimeError("oracle template not found")
        pyxtal_result = pyxtal_result_for_graph(graph_idx0)
        free_vars = recover_template_free_vars_from_pyxtal_result(template, pyxtal_result).to(device=runner.device, dtype=dtype)
        expansion = expand_wyckoff_template_torch(template=template, free_vars=free_vars)
        source_frac = expansion.frac_coords.to(device=runner.device, dtype=dtype)
        source_z = expansion.atomic_numbers.to(device=runner.device, dtype=torch.long)
        raw = species_matched_torus_metrics(source_frac, source_z, target_frac, target_z)
        best = best_tau_metrics(source_frac, source_z, target_frac, target_z)
        cell = torch.as_tensor(np.array(standardized.lattice.matrix, dtype=float), device=runner.device, dtype=dtype)
        k = cell_to_k(cell, eps=1e-8)
        lattice_k_loss = float((k - k).square().mean().item())
        min_pair = min_pair_distance_structure(standardized)
        passed = raw["assignment_cost"] < 1e-5 or best["assignment_cost"] < 1e-5
        rows.append({
            "test": "D_materialization_identity",
            "graph": graph_id,
            "coord_loss_raw": raw["assignment_cost"],
            "coord_loss_best_tau": best["assignment_cost"],
            "frac_rmse_raw": raw["frac_rmse"],
            "frac_rmse_best_tau": best["frac_rmse"],
            "best_tau": best["best_tau"],
            "lattice_k_loss": lattice_k_loss,
            "volume_loss": 0.0,
            "min_pair": min_pair,
            "sg_detected": detect_sg(standardized, TEST_ALGO10.symprec, TEST_ALGO10.angle_tolerance),
            "structure_match": StructureMatcher().fit(standardized, standardized),
            "PASS": passed,
            "status": "PASS" if passed else "FAIL",
            "failure_category": None if passed else "ORIGIN_OR_CHART_MISMATCH",
            "action_needed": None if passed else "debug origin settings, equivalent Wyckoff representatives, and frame conversion",
        })
    except Exception as exc:
        rows.append(error_row("D_materialization_identity", graph_id, exc, "ORIGIN_OR_CHART_MISMATCH"))
df_D = dataframe_result("D_materialization_identity", rows)
df_D


,test,graph,coord_loss_raw,coord_loss_best_tau,frac_rmse_raw,frac_rmse_best_tau,best_tau,lattice_k_loss,volume_loss,min_pair,sg_detected,structure_match,PASS,status,failure_category,action_needed
0,D_materialization_identity,1,1.562500e-02,0.000000e+00,1.250000e-01,0.000000e+00,"(0.125, 0.125, 0.125)",0.0,0.0,2.664643,227,True,True,PASS,None,None
1,D_materialization_identity,2,2.035409e-16,2.035409e-16,1.426678e-08,1.426678e-08,"(0.0, 0.0, 0.0)",0.0,0.0,2.036511,4,True,True,PASS,None,None
2,D_materialization_identity,3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,"(0.0, 0.0, 0.0)",0.0,0.0,2.911619,194,True,True,PASS,None,None


## 7. Test E — DiffCSP++ `k`-basis lattice sanity

**Actually tests:** Checks the DiffCSP++ k-coordinate lattice basis and crystal-family constraints independently of CASAL.


In [48]:
rows = []
for graph_idx0, graph_id in zip(GRAPH_INDICES, GRAPH_IDS):
    try:
        g = graph_tensors(graph_idx0)
        _, standardized = standard_graph_structure(graph_idx0)
        cell = torch.as_tensor(np.array(standardized.lattice.matrix, dtype=float), device=runner.device, dtype=dtype)
        k = cell_to_k(cell, eps=1e-8)
        constraint = space_group_k_constraint(space_group_number=g["sg"], device=runner.device, dtype=dtype)
        fixed_constraint_error = float(((k - constraint.target) * constraint.mask).abs().max().detach().item())
        cell_recon = k_to_cell_matrix(k)
        gram = cell @ cell.transpose(-1, -2)
        gram_recon = cell_recon @ cell_recon.transpose(-1, -2)
        gram_relative_error = float((torch.linalg.norm(gram - gram_recon) / torch.linalg.norm(gram).clamp_min(1e-8)).detach().item())
        raw_cell_relative_error = float((torch.linalg.norm(cell - cell_recon) / torch.linalg.norm(cell).clamp_min(1e-8)).detach().item())
        k_projected = k.clone()
        k_projected = k_projected * (1.0 - constraint.mask) + constraint.target * constraint.mask
        cell_projected = k_to_cell_matrix(k_projected)
        positive_volume = bool(torch.det(cell_projected).abs().item() > 1e-8)
        fam_ok = family_constraints_ok(g["sg"], cell_projected)
        passed = fixed_constraint_error < 1e-3 and gram_relative_error < 1e-5 and positive_volume and fam_ok
        rows.append({
            "test": "E_k_basis_lattice",
            "graph": graph_id,
            "sg": g["sg"],
            "family": family_name(g["sg"]),
            "k": [float(v) for v in k.detach().cpu().reshape(-1).tolist()],
            "fixed_constraint_error": fixed_constraint_error,
            "cell_reconstruction_gram_error": gram_relative_error,
            "raw_cell_relative_error_orientation_sensitive": raw_cell_relative_error,
            "volume_original": float(torch.det(cell).abs().item()),
            "volume_reconstructed": float(torch.det(cell_recon).abs().item()),
            "projected_family_ok": fam_ok,
            "positive_volume_after_mask": positive_volume,
            "PASS": passed,
            "status": "PASS" if passed else "FAIL",
            "failure_category": None if passed else "K_BASIS_FAILURE",
            "action_needed": None if passed else "debug cell_to_k/k_to_cell_matrix or the space_group_k_constraint mask",
        })
    except Exception as exc:
        rows.append(error_row("E_k_basis_lattice", graph_id, exc, "K_BASIS_FAILURE"))
df_E = dataframe_result("E_k_basis_lattice", rows)
df_E


,test,graph,sg,family,k,fixed_constraint_error,cell_reconstruction_gram_error,raw_cell_relative_error_orientation_sensitive,volume_original,volume_reconstructed,projected_family_ok,positive_volume_after_mask,PASS,status,failure_category,action_needed
0,E_k_basis_lattice,1,227,cubic,"[0.0, 0.0, 9.799677513910865e-09, 0.0, 0.0, 2....",9.799678e-09,6.715710e-08,6.326827e-08,428.106873,428.106812,True,True,True,PASS,None,None
1,E_k_basis_lattice,2,4,monoclinic,"[0.0, -0.0004923938540741801, 0.0, -0.08761519...",0.000000e+00,5.371198e-07,1.721122e+00,154.887314,154.887360,True,True,True,PASS,None,None
2,E_k_basis_lattice,3,194,hexagonal_trigonal,"[-0.27465304732322693, 0.0, 0.0, 0.0, 0.055292...",2.980232e-08,1.070762e-07,6.685109e-01,149.449341,149.449341,True,True,True,PASS,None,None


## 8. Test F — fixed-template SSVD projection to ground truth

**Actually tests:** Checks the fixed-template SSVD operator on ground truth over the full origin grid; this is the strict oracle projection identity test.


In [49]:

rows = []
for graph_idx0, graph_id in zip(GRAPH_INDICES, GRAPH_IDS):
    try:
        g = graph_tensors(graph_idx0)
        cell = cell_from_l(g["l"], g["num_atoms"]).to(device=runner.device, dtype=dtype)
        chart_frac, chart_h, chart_cell, chart_k = _build_chart_target(
            frac_coords=g["pos"],
            atomic_numbers=g["h"],
            cell_matrix=cell,
            requested_sg=g["sg"],
            symprec=ORACLE_ALGO10.symprec,
            angle_tolerance=ORACLE_ALGO10.angle_tolerance,
        )
        _, standardized = standard_graph_structure(graph_idx0)
        standardized_cell = torch.as_tensor(np.array(standardized.lattice.matrix, dtype=float), device=runner.device, dtype=dtype)
        standardized_k = cell_to_k(standardized_cell, eps=1e-8)
        chart_vs_standardized_k_loss = float((chart_k - standardized_k).square().mean().detach().item()) if chart_k.shape == standardized_k.shape else float("nan")
        state = ssvd_state_for_graph(graph_idx0, require_oracle=True)
        zero_tau = torch.zeros((1, 3), device=runner.device, dtype=dtype)
        initial_cfg = dataclasses.replace(ORACLE_ALGO10.ssvd_projection_config(), max_steps=0, random_restarts=1)
        zero_initial = fixed_template_ssvd_project(
            template_state=state,
            y_f=chart_frac,
            y_k=chart_k,
            y_h=chart_h.to(device=runner.device, dtype=torch.long),
            tau0=zero_tau,
            metric=ORACLE_ALGO10.projection_metric(),
            config=initial_cfg,
        )
        zero_final = fixed_template_ssvd_project(
            template_state=state,
            y_f=chart_frac,
            y_k=chart_k,
            y_h=chart_h.to(device=runner.device, dtype=torch.long),
            tau0=zero_tau,
            metric=ORACLE_ALGO10.projection_metric(),
            config=ORACLE_ALGO10.ssvd_projection_config(),
        )
        best, all_results, errors = fixed_template_best_over_origin_grid(
            state,
            y_f=chart_frac,
            y_k=chart_k,
            y_h=chart_h,
            config=ORACLE_ALGO10,
        )
        free_dim = int(state.template.total_free_dims)
        lattice_dim = int(state.lattice_free_vars.numel())
        expected_rank_upper = free_dim + lattice_dim + 3
        line_search_ok = best.ssvd_line_search_failures <= max(1, best.ssvd_steps // 2)
        condition_ok = math.isfinite(best.ssvd_condition_number) and best.ssvd_condition_number < SSVD_CONDITION_TOL
        passed = (
            math.isfinite(best.objective)
            and best.objective < GT_PROJECTION_OBJECTIVE_TOL
            and best.coord_loss < GT_COORD_LOSS_TOL
            and best.lattice_k_loss < GT_K_LOSS_TOL
            and line_search_ok
            and condition_ok
        )
        rows.append({
            "test": "F_fixed_template_ssvd",
            "graph": graph_id,
            "template_labels": ",".join(f"{s.atomic_number}@{s.label}" for s in state.template.site_templates),
            "objective_zero_tau_initial": zero_initial.objective,
            "objective_zero_tau_final": zero_final.objective,
            "objective_best_tau": best.objective,
            "coord_loss_zero_tau": zero_final.coord_loss,
            "coord_loss_best_tau": best.coord_loss,
            "lattice_k_loss_zero_tau": zero_final.lattice_k_loss,
            "lattice_k_loss_best_tau": best.lattice_k_loss,
            "origin_grid_count": len(all_results) + len(errors),
            "origin_grid_failures": len(errors),
            "best_tau": tuple(float(v) for v in best.tau.detach().cpu().reshape(-1).tolist()),
            "chart_atom_count": int(chart_h.numel()),
            "standardized_atom_count": int(len(standardized)),
            "chart_vs_standardized_k_loss": chart_vs_standardized_k_loss,
            "num_steps": best.ssvd_steps,
            "num_accepted_line_search": best.ssvd_line_search_accepts,
            "num_line_search_failures": best.ssvd_line_search_failures,
            "num_clipped_steps": best.ssvd_clip_count,
            "line_search_ok": line_search_ok,
            "jacobian_rank": best.ssvd_rank,
            "expected_rank_upper": expected_rank_upper,
            "jacobian_condition": best.ssvd_condition_number,
            "condition_ok": condition_ok,
            "min_sigma": best.ssvd_min_sigma,
            "max_sigma": best.ssvd_max_sigma,
            "PASS": passed,
            "status": "PASS" if passed else "FAIL",
            "failure_category": None if passed else "SSVD_NOT_IDENTITY_ON_GT",
            "action_needed": None if passed else "inspect origin shifts, chart gauge/rank, line search failures, and lattice-k residual",
        })
    except Exception as exc:
        rows.append(error_row("F_fixed_template_ssvd", graph_id, exc, "SSVD_NOT_IDENTITY_ON_GT"))
df_F = dataframe_result("F_fixed_template_ssvd", rows)
df_F


,test,graph,template_labels,objective_zero_tau_initial,objective_zero_tau_final,objective_best_tau,coord_loss_zero_tau,coord_loss_best_tau,lattice_k_loss_zero_tau,lattice_k_loss_best_tau,...,jacobian_rank,expected_rank_upper,jacobian_condition,condition_ok,min_sigma,max_sigma,PASS,status,failure_category,action_needed
0,F_fixed_template_ssvd,1,"70@8a,77@16d",2.724359e-02,4.775755e-14,4.757536e-14,1.973730e-16,0.000000e+00,2.675919e-15,2.675919e-15,...,4,4,4.898980,True,1.0,4.898980,True,PASS,None,None
1,F_fixed_template_ssvd,2,"26@2a,26@2a,26@2a,27@2a,27@2a,27@2a,5@2a,5@2a",1.809252e-16,6.579099e-17,6.579099e-17,7.401487e-17,7.401487e-17,0.000000e+00,0.000000e+00,...,30,31,4.242642,True,1.0,4.242641,True,PASS,None,None
2,F_fixed_template_ssvd,3,"72@2d,22@6h",2.960595e-17,2.960595e-17,2.960595e-17,0.000000e+00,0.000000e+00,1.480297e-16,1.480297e-16,...,6,6,4.898980,True,1.0,4.898980,True,PASS,None,None


## 9. Test G — wrong-template stress test

**Actually tests:** Checks whether non-oracle/wrong Wyckoff templates are separated from the oracle, or only tie when structurally equivalent.


In [50]:

rows = []
matcher = StructureMatcher(primitive_cell=False, scale=True, attempt_supercell=False)
for graph_idx0, graph_id in zip(GRAPH_INDICES, GRAPH_IDS):
    try:
        g = graph_tensors(graph_idx0)
        cell = cell_from_l(g["l"], g["num_atoms"]).to(device=runner.device, dtype=dtype)
        chart_frac, chart_h, chart_cell, chart_k = _build_chart_target(
            frac_coords=g["pos"],
            atomic_numbers=g["h"],
            cell_matrix=cell,
            requested_sg=g["sg"],
            symprec=ORACLE_ALGO10.symprec,
            angle_tolerance=ORACLE_ALGO10.angle_tolerance,
        )
        oracle_sig = oracle_signature_for_graph(graph_idx0)
        oracle_state = ssvd_state_for_graph(graph_idx0, require_oracle=True)
        wrong_pool = initialize_constrained_template_states(
            reference_frac_coords=g["pos"], atomic_numbers=g["h"], cell_matrix=cell,
            space_group_number=g["sg"], standardization="conventional",
            symprec=NONORACLE_ALGO10.symprec, angle_tolerance=NONORACLE_ALGO10.angle_tolerance,
            max_templates=CPU_WRONG_TEMPLATE_MAX if M1_CPU_PROFILE else ORACLE_TEMPLATE_MAX_TEMPLATES,
            template_eval_limit=CPU_WRONG_TEMPLATE_EVAL_LIMIT if M1_CPU_PROFILE else min(ORACLE_ALGO10.template_eval_limit, 32),
            quick_templates=NONORACLE_ALGO10.quick_templates,
            top_k=CPU_WRONG_TEMPLATE_EVAL_LIMIT if M1_CPU_PROFILE else min(ORACLE_ALGO10.template_eval_limit, 32),
            template_prior=template_prior, template_prior_weight=NONORACLE_ALGO10.template_prior_weight,
            debug_template_candidates=False,
        )
        wrong_states = [s for s in wrong_pool if flatten_site_signature(s.template) != oracle_sig][:3]
        states = [oracle_state] + wrong_states
        projected = []
        for state in states:
            best, _all_results, errors = fixed_template_best_over_origin_grid(
                state,
                y_f=chart_frac,
                y_k=chart_k,
                y_h=chart_h,
                config=ORACLE_ALGO10 if flatten_site_signature(state.template) == oracle_sig else BASE_TEST_ALGO10,
            )
            projected.append((state, best, errors))
        oracle_projection = projected[0][1]
        oracle_min = float(oracle_projection.objective)
        oracle_structure = structure_from_fixed_projection(oracle_projection)
        for state, result, errors in projected:
            is_oracle = flatten_site_signature(state.template) == oracle_sig
            structure = structure_from_fixed_projection(result)
            equivalent_to_oracle = bool(matcher.fit(structure, oracle_structure))
            separated = bool(float(result.objective) > oracle_min + WRONG_TEMPLATE_MARGIN)
            dangerous_tie = bool((not is_oracle) and (not separated) and (not equivalent_to_oracle))
            classification = "oracle" if is_oracle else ("separated" if separated else ("equivalent" if equivalent_to_oracle else "dangerous_tie"))
            passed = bool(is_oracle or separated or equivalent_to_oracle)
            rows.append({
                "test": "G_wrong_template_stress",
                "graph": graph_id,
                "template_label": ",".join(f"{s.atomic_number}@{s.label}" for s in state.template.site_templates),
                "is_oracle_signature": is_oracle,
                "classification": classification,
                "dangerous_tie": dangerous_tie,
                "oracle_min_residual": oracle_min,
                "final_residual": result.objective,
                "residual_margin_vs_oracle": float(result.objective) - oracle_min,
                "coord_loss_best_tau": result.coord_loss,
                "lattice_k_loss_best_tau": result.lattice_k_loss,
                "best_tau": tuple(round(float(v), 4) for v in result.tau.reshape(-1).detach().cpu().tolist()),
                "structure_match_to_oracle": equivalent_to_oracle,
                "min_pair": result.min_pair_distance,
                "origin_grid_count": len(ssvd_origin_grid),
                "origin_grid_failures": len(errors),
                "PASS": passed,
                "status": "PASS" if passed else "FAIL",
                "failure_category": None if passed else "DANGEROUS_TEMPLATE_TIE",
                "action_needed": None if passed else "projection metric cannot distinguish this wrong Wyckoff template; add discrete move support or stronger orbit metric",
            })
    except Exception as exc:
        rows.append(error_row("G_wrong_template_stress", graph_id, exc, "WRONG_TEMPLATE_STRESS_ERROR"))
df_G = dataframe_result("G_wrong_template_stress", rows)
df_G


,test,graph,template_label,is_oracle_signature,classification,dangerous_tie,oracle_min_residual,final_residual,residual_margin_vs_oracle,coord_loss_best_tau,lattice_k_loss_best_tau,best_tau,structure_match_to_oracle,min_pair,origin_grid_count,origin_grid_failures,PASS,status,failure_category,action_needed
0,G_wrong_template_stress,1,"70@8a,77@16d",True,oracle,False,4.757536e-14,4.757536e-14,0.000000,0.000000e+00,2.675919e-15,"(0.875, 0.875, 0.375)",True,2.664643,216,0,True,PASS,None,None
1,G_wrong_template_stress,1,"70@8b,77@16c",False,equivalent,False,4.757536e-14,4.757536e-14,0.000000,0.000000e+00,2.675919e-15,"(0.875, 0.375, 0.375)",True,2.664643,216,0,True,PASS,None,None
2,G_wrong_template_stress,1,"70@8a,77@16c",False,separated,False,4.757536e-14,1.495726e-02,0.014957,1.620370e-02,2.675919e-15,"(0.7083, 0.0417, 0.875)",False,1.631754,216,0,True,PASS,None,None
3,G_wrong_template_stress,1,"70@8b,77@16d",False,separated,False,4.757536e-14,1.495726e-02,0.014957,1.620370e-02,2.675919e-15,"(0.0417, 0.2083, 0.875)",False,1.631753,216,0,True,PASS,None,None
4,G_wrong_template_stress,2,"26@2a,26@2a,26@2a,27@2a,27@2a,27@2a,5@2a,5@2a",True,oracle,False,6.579099e-17,6.579099e-17,0.000000,7.401487e-17,0.000000e+00,"(0.0, 0.0, 0.0)",True,2.036511,216,0,True,PASS,None,None
5,G_wrong_template_stress,3,"72@2d,22@6h",True,oracle,False,2.960595e-17,2.960595e-17,0.000000,0.000000e+00,1.480297e-16,"(0.0, 0.0, 0.0)",True,2.911619,216,0,True,PASS,None,None
6,G_wrong_template_stress,3,"72@2a,22@2d,22@2c,22@2b",False,separated,False,2.960595e-17,1.703716e-02,0.017037,2.129645e-02,1.480297e-16,"(0.625, 0.625, 0.4375)",False,1.187805,216,0,True,PASS,None,None
7,G_wrong_template_stress,3,"72@2a,22@4e,22@2b",False,separated,False,2.960595e-17,3.267103e-01,0.326710,5.039683e-02,2.871887e-04,"(0.0, 0.0, 0.0312)",False,0.511736,216,0,True,PASS,None,None
8,G_wrong_template_stress,3,"72@2a,22@4e,22@2c",False,separated,False,2.960595e-17,6.165859e-02,0.061659,3.345310e-02,2.100108e-04,"(0.5, 0.0, 0.5358)",False,0.841973,216,0,True,PASS,None,None


## 10. Test H — full projection over `(W, theta, tau, pi)`

**Actually tests:** Checks deployable non-oracle full chart projection on ground truth and KLDM samples: SG validity is necessary, but identity/residual/volume/jump constraints are also enforced.


In [51]:

rows = []
projection_sources = [("gt", None)]
if RUN_MODEL_SAMPLING_TESTS:
    try:
        projection_sources.append(("kldm", get_kldm_sample_once()))
    except Exception as exc:
        rows.append(error_row("H_full_projection", "all", exc, "KLDM_SAMPLE_ERROR"))
else:
    print("H_full_projection: KLDM-source projection skipped because RUN_MODEL_SAMPLING_TESTS=False; GT identity still runs.")

for target_type, source in projection_sources:
    for graph_idx0, graph_id in zip(GRAPH_INDICES, GRAPH_IDS):
        try:
            g = graph_tensors(graph_idx0, source=source)
            projection = _project_graph_to_chart(
                graph_idx=graph_idx0,
                requested_sg=g["sg"],
                target_pos=g["pos"],
                target_l=g["l"],
                target_h=g["h"],
                lattice_transform=runner.lattice_transform,
                config=NONORACLE_ALGO10,
                template_prior=template_prior,
                oracle_reference_structure=None,
            )
            volume_ratio = projection_volume_ratio(graph_idx0, projection, source=source)
            volume_ok = volume_ratio_ok(NONORACLE_ALGO10, volume_ratio)
            projection_jump_frac = math.sqrt(max(float(projection.coord_loss), 0.0))
            projection_jump_k = math.sqrt(max(float(projection.lattice_k_loss), 0.0))
            if target_type == "gt":
                residual_ok = (
                    projection.objective < GT_PROJECTION_OBJECTIVE_TOL
                    and projection.coord_loss < GT_COORD_LOSS_TOL
                    and projection.lattice_k_loss < GT_K_LOSS_TOL
                )
                jump_ok = residual_ok
            else:
                residual_ok = projection.objective < KLDM_PROJECTION_OBJECTIVE_TOL
                jump_ok = projection_jump_frac <= KLDM_MAX_PROJECTION_JUMP_FRAC and projection_jump_k <= KLDM_MAX_PROJECTION_JUMP_K
            passed = bool(
                projection.requested_sg_match
                and projection.composition_match
                and projection.min_pair_distance >= NONORACLE_ALGO10.hard_min_distance
                and volume_ok
                and residual_ok
                and jump_ok
            )
            rows.append({
                "test": "H_full_projection",
                "graph": graph_id,
                "target_type": target_type,
                "template_labels": projection.template_labels,
                "best_tau": tuple(round(float(v), 4) for v in projection.tau.reshape(-1).detach().cpu().tolist()),
                "objective": projection.objective,
                "coord_loss": projection.coord_loss,
                "lattice_k_loss": projection.lattice_k_loss,
                "projection_jump_frac": projection_jump_frac,
                "projection_jump_k": projection_jump_k,
                "residual_ok": residual_ok,
                "volume_loss": projection.volume_loss,
                "steric_loss": projection.steric_loss,
                "min_pair": projection.min_pair_distance,
                "volume_ratio": volume_ratio,
                "volume_ratio_ok": volume_ok,
                "sg_detected": projection.sg_detected,
                "requested_sg_match": projection.requested_sg_match,
                "composition_match": projection.composition_match,
                "PASS": passed,
                "status": "PASS" if passed else "FAIL",
                "failure_category": None if passed else ("GT_PROJECTION_NOT_IDENTITY" if target_type == "gt" else "KLDM_PROJECTION_TOO_VIOLENT"),
                "action_needed": None if passed else "inspect non-oracle template search, tau/orbit selection, residuals, volume ratio, and projection jump",
            })
        except Exception as exc:
            rows.append(error_row("H_full_projection", graph_id, exc, "PROJECTION_UNSTABLE"))
df_H = dataframe_result("H_full_projection", rows)
df_H


,test,graph,target_type,template_labels,best_tau,objective,coord_loss,lattice_k_loss,projection_jump_frac,projection_jump_k,...,min_pair,volume_ratio,volume_ratio_ok,sg_detected,requested_sg_match,composition_match,PASS,status,failure_category,action_needed
0,H_full_projection,1,gt,"70@8b,77@16c","(0.875, 0.875, 0.875)",4.757536e-14,0.000000e+00,2.675919e-15,0.000000e+00,5.172928e-08,...,2.664643,1.000000,True,227,True,True,True,PASS,None,None
1,H_full_projection,2,gt,"26@2a,26@2a,26@2a,27@2a,27@2a,27@2a,5@2a,5@2a","(0.0, 0.0022, 0.0)",2.631640e-16,2.960595e-16,0.000000e+00,1.720638e-08,0.000000e+00,...,2.036511,1.000000,True,4,True,True,True,PASS,None,None
2,H_full_projection,3,gt,"72@2c,22@6h","(0.0, 0.0, 0.5)",5.033011e-16,5.921190e-16,1.480297e-16,2.433349e-08,1.216675e-08,...,2.911619,1.000000,True,194,True,True,True,PASS,None,None
3,H_full_projection,1,kldm,"70@8b,77@16c","(0.071, 0.1081, 0.7266)",4.374089e-04,7.022027e-09,2.735279e-05,8.379754e-05,5.229990e-03,...,2.655575,1.000000,True,227,True,True,True,PASS,None,None
4,H_full_projection,2,kldm,"26@2a,26@2a,26@2a,27@2a,27@2a,27@2a,5@2a,5@2a","(0.9032, 0.893, 0.8802)",1.115314e-02,1.195119e-02,3.262938e-05,1.093215e-01,5.712213e-03,...,1.042896,1.000001,True,4,True,True,True,PASS,None,None
5,H_full_projection,3,kldm,"72@2d,22@4f,22@2c","(0.087, 0.4646, 0.6285)",1.758519e-03,1.281429e-03,6.279293e-05,3.579706e-02,7.924199e-03,...,1.957835,1.000000,True,194,True,True,True,PASS,None,None


## 11. Test I — CASAL split-state sanity in `(f, k)`

**Actually tests:** Checks that CASAL maintains separate fractional and k-space split variables and duals with compatible graph-local shapes.


In [52]:
rows = []
for graph_idx0, graph_id in zip(GRAPH_INDICES, GRAPH_IDS):
    try:
        g = graph_tensors(graph_idx0)
        state = _init_graph_state(
            graph_idx=graph_idx0, requested_sg=g["sg"], pos=g["pos"], l=g["l"], h=g["h"],
            lattice_transform=runner.lattice_transform, config=NONORACLE_ALGO10, template_prior=template_prior,
            oracle_reference_structure=None,
        )
        x_k = _l_to_k(l=g["l"], num_atoms=g["num_atoms"], lattice_transform=runner.lattice_transform)
        xz_frac = wrap_delta(g["pos"] - state.z_pos)
        xz_k = x_k - state.z_k
        mu_f_clip_fraction = float((state.mu_f.abs() >= NONORACLE_ALGO10.casal_mu_clip).float().mean().item()) if NONORACLE_ALGO10.casal_mu_clip > 0 else 0.0
        mu_k_clip_fraction = float((state.mu_k.abs() >= NONORACLE_ALGO10.casal_mu_clip).float().mean().item()) if NONORACLE_ALGO10.casal_mu_clip > 0 else 0.0
        xz_k_residual_norm = float(torch.linalg.norm(xz_k.reshape(-1)).item())
        graph3_k_residual_warn = bool(graph_id == 3 and xz_k_residual_norm > GRAPH3_K_RESIDUAL_WARN)
        passed = (
            tuple(g["pos"].shape) == tuple(state.z_pos.shape) == tuple(state.mu_f.shape)
            and tuple(x_k.shape) == tuple(state.z_k.shape) == tuple(state.mu_k.shape)
            and torch.isfinite(xz_frac).all().item()
            and torch.isfinite(xz_k).all().item()
        )
        rows.append({
            "test": "I_casal_split_state",
            "graph": graph_id,
            "x_f_shape": tuple(g["pos"].shape),
            "z_f_shape": tuple(state.z_pos.shape),
            "mu_f_shape": tuple(state.mu_f.shape),
            "x_k_shape": tuple(x_k.shape),
            "z_k_shape": tuple(state.z_k.shape),
            "mu_k_shape": tuple(state.mu_k.shape),
            "xz_frac_residual_norm": float(torch.linalg.norm(xz_frac.reshape(-1)).item()),
            "xz_k_residual_norm": xz_k_residual_norm,
            "graph3_k_residual_warn": graph3_k_residual_warn,
            "graph3_k_residual_warn_threshold": GRAPH3_K_RESIDUAL_WARN,
            "mu_f_norm": float(torch.linalg.norm(state.mu_f.reshape(-1)).item()),
            "mu_k_norm": float(torch.linalg.norm(state.mu_k.reshape(-1)).item()),
            "mu_f_clip_fraction": mu_f_clip_fraction,
            "mu_k_clip_fraction": mu_k_clip_fraction,
            "PASS": passed,
            "status": "PASS" if passed else "FAIL",
            "failure_category": None if passed else "DUAL_OR_STATE_SHAPE_FAILURE",
            "action_needed": None if passed and not graph3_k_residual_warn else ("watch graph 3 mu_k, volume, and projection jumps in sampling" if graph3_k_residual_warn else "inspect (f,k) state packing and graph slicing"),
        })
    except Exception as exc:
        rows.append(error_row("I_casal_split_state", graph_id, exc, "DUAL_OR_STATE_SHAPE_FAILURE"))
df_I = dataframe_result("I_casal_split_state", rows)
df_I


,test,graph,x_f_shape,z_f_shape,mu_f_shape,x_k_shape,z_k_shape,mu_k_shape,xz_frac_residual_norm,xz_k_residual_norm,graph3_k_residual_warn,graph3_k_residual_warn_threshold,mu_f_norm,mu_k_norm,mu_f_clip_fraction,mu_k_clip_fraction,PASS,status,failure_category,action_needed
0,I_casal_split_state,1,"(6, 3)","(6, 3)","(6, 3)","(6,)","(6,)","(6,)",0.530330,6.681379e-07,False,0.25,0.0,0.0,0.0,0.0,True,PASS,None,None
1,I_casal_split_state,2,"(16, 3)","(16, 3)","(16, 3)","(6,)","(6,)","(6,)",0.879446,9.847881e-04,False,0.25,0.0,0.0,0.0,0.0,True,PASS,None,None
2,I_casal_split_state,3,"(8, 3)","(8, 3)","(8, 3)","(6,)","(6,)","(6,)",0.933661,4.057415e-01,True,0.25,0.0,0.0,0.0,0.0,True,PASS,None,"watch graph 3 mu_k, volume, and projection jum..."


## 12. Test J — CASAL-lite one-step projection/coupling

**Actually tests:** Checks one deployable non-oracle CASAL-lite step using the returned projection object and the actual x-z residual.


In [53]:

rows = []
if not RUN_MODEL_SAMPLING_TESTS:
    df_J = skipped_df("J_one_step_casal_lite", "RUN_MODEL_SAMPLING_TESTS=False")
else:
    try:
        kldm_source = get_kldm_sample_once()
        for graph_idx0, graph_id in zip(GRAPH_INDICES, GRAPH_IDS):
            try:
                g = graph_tensors(graph_idx0, source=kldm_source)
                state = _init_graph_state(
                    graph_idx=graph_idx0, requested_sg=g["sg"], pos=g["pos"], l=g["l"], h=g["h"],
                    lattice_transform=runner.lattice_transform, config=NONORACLE_ALGO10, template_prior=template_prior,
                    oracle_reference_structure=None,
                )
                objective_before = state.projection.objective
                x_k_before = _l_to_k(l=g["l"], num_atoms=g["num_atoms"], lattice_transform=runner.lattice_transform)
                residual_before = float(torch.linalg.norm(torch.cat([wrap_delta(g["pos"] - state.z_pos).reshape(-1), (x_k_before - state.z_k).reshape(-1)])).item())
                x_pos, x_l, projection, projection_error, r_coord, r_k, mu_coord, mu_k = _casal_step_graph(
                    graph_idx=graph_idx0, requested_sg=g["sg"], x_pos=g["pos"], x_l=g["l"], h=g["h"],
                    graph_state=state, rho=NONORACLE_ALGO10.casal_rho_end, tau_step=1e-4,
                    should_project=True, lattice_transform=runner.lattice_transform, config=NONORACLE_ALGO10,
                    template_prior=template_prior,
                    oracle_reference_structure=None,
                )
                residual_after = math.sqrt(r_coord * r_coord + r_k * r_k)
                residual_improved = residual_after <= residual_before + 1e-8
                objective_after = None if projection is None else projection.objective
                sg_match_after = False if projection is None else projection.requested_sg_match
                min_pair_after = float("nan") if projection is None else projection.min_pair_distance
                passed = bool(
                    projection_error is None
                    and projection is not None
                    and projection.requested_sg_match
                    and projection.min_pair_distance >= NONORACLE_ALGO10.hard_min_distance
                    and math.isfinite(residual_after)
                    and residual_improved
                )
                rows.append({
                    "test": "J_one_step_casal_lite",
                    "graph": graph_id,
                    "objective_before": objective_before,
                    "objective_after": objective_after,
                    "xz_residual_before": residual_before,
                    "xz_residual_after": residual_after,
                    "residual_improved_or_stable": residual_improved,
                    "mu_norm_after": math.sqrt(mu_coord * mu_coord + mu_k * mu_k),
                    "sg_match_after": sg_match_after,
                    "min_pair_after": min_pair_after,
                    "projection_error": None if projection_error is None else repr(projection_error),
                    "PASS": passed,
                    "status": "PASS" if passed else "FAIL",
                    "failure_category": None if passed else "CASAL_ONE_STEP_FAILURE",
                    "action_needed": None if passed else "inspect returned projection, residual growth, rho/tau, and dual residuals",
                })
            except Exception as exc:
                rows.append(error_row("J_one_step_casal_lite", graph_id, exc, "CASAL_ONE_STEP_FAILURE"))
        df_J = dataframe_result("J_one_step_casal_lite", rows)
    except Exception as exc:
        df_J = dataframe_result("J_one_step_casal_lite", [error_row("J_one_step_casal_lite", "all", exc, "KLDM_SAMPLE_ERROR")])
df_J


,test,graph,objective_before,objective_after,xz_residual_before,xz_residual_after,residual_improved_or_stable,mu_norm_after,sg_match_after,min_pair_after,projection_error,PASS,status,failure_category,action_needed
0,J_one_step_casal_lite,1,0.000437,4.898914e-10,1.210446,1.210265,True,1.056324,True,2.655574,None,True,PASS,None,None
1,J_one_step_casal_lite,2,0.010175,2.631640e-16,1.328201,1.496120,False,1.423477,True,1.569395,None,False,FAIL,CASAL_ONE_STEP_FAILURE,"inspect returned projection, residual growth, ..."
2,J_one_step_casal_lite,3,0.001759,2.960595e-17,1.219640,1.219454,True,1.067333,True,1.957832,None,True,PASS,None,None


## 13. Test K — projection interval, start-time, and rho sweep

**Actually tests:** Optionally sweeps projection timing, rho, and tau against a real final-projection baseline; skipped by default because it is expensive.


In [54]:

if not RUN_SWEEP_K:
    df_K = skipped_df("K_schedule_sweep", "RUN_SWEEP_K=False; enable for expensive Algorithm10 schedule sweep", graphs=["sweep"])
else:
    rows = []
    try:
        baseline_source = get_kldm_sample_once()
        baseline_rows = project_source_rows_to_chart(
            "K_schedule_sweep_baseline",
            baseline_source,
            config=NONORACLE_ALGO10,
            method="FinalProjectionBaseline@1#1",
        )
        final_projection_valid_rate = float(np.mean([bool(r.get("valid")) for r in baseline_rows]))
        final_projection_sg_rate = float(np.mean([bool(r.get("requested_sg_match")) for r in baseline_rows]))
        final_projection_min_pairs = [r.get("min_pair_distance") for r in baseline_rows if r.get("min_pair_distance") is not None]
        final_projection_mean_min_pair = float(np.nanmean(final_projection_min_pairs)) if final_projection_min_pairs else np.nan
    except Exception as exc:
        baseline_rows = []
        final_projection_valid_rate = 0.0
        final_projection_sg_rate = 0.0
        final_projection_mean_min_pair = np.nan
        rows.append(error_row("K_schedule_sweep", "baseline", exc, "FINAL_PROJECTION_BASELINE_ERROR"))

    projection_start_fraction_values = [0.50, 0.65, 0.75, 0.85]
    projection_interval_values = [1, 5, 10, 20]
    rho_end_values = [1.0, 2.0, 5.0, 10.0]
    tau_scale_values = [0.01, 0.025, 0.05, 0.1]
    combos = list(itertools.product(projection_start_fraction_values, projection_interval_values, rho_end_values, tau_scale_values))[:MAX_SWEEP_COMBINATIONS]
    for start_fraction, interval, rho_end, tau_scale in combos:
        try:
            cfg = sampler_cfg_with_algorithm10(
                sampler_cfgs["Algorithm10_CASAL_chart"],
                NONORACLE_ALGO10,
                n_steps=min(int(sampler_cfgs["Algorithm10_CASAL_chart"].get("n_steps", 120)), 120),
                updates={
                    "projection_start_fraction": float(start_fraction),
                    "projection_interval": int(interval),
                    "casal_rho_end": float(rho_end),
                    "casal_tau_scale": float(tau_scale),
                },
            )
            set_seed(SAMPLE_SEED)
            pos, _v, l, h = runner._sample_batch(batch, sampler_cfg=cfg)
            eval_rows = evaluate_prediction_rows("K_schedule_sweep", pos, l, h, method=f"start={start_fraction},interval={interval},rho={rho_end},tau={tau_scale}")
            sg_rate = float(np.mean([bool(r.get("requested_sg_match")) for r in eval_rows]))
            valid_rate = float(np.mean([bool(r.get("valid")) for r in eval_rows]))
            min_pairs = [r.get("min_pair_distance") for r in eval_rows if r.get("min_pair_distance") is not None]
            mean_min_pair = float(np.nanmean(min_pairs)) if min_pairs else np.nan
            passed = bool(
                valid_rate >= final_projection_valid_rate - 0.05
                and sg_rate >= final_projection_sg_rate - 0.05
                and math.isfinite(mean_min_pair)
                and mean_min_pair >= NONORACLE_ALGO10.hard_min_distance
            )
            rows.append({
                "test": "K_schedule_sweep",
                "graph": "all",
                "projection_start_fraction": start_fraction,
                "projection_interval": interval,
                "rho_end": rho_end,
                "tau_scale": tau_scale,
                "sg_match_rate": sg_rate,
                "valid_rate": valid_rate,
                "mean_min_pair": mean_min_pair,
                "final_projection_sg_rate": final_projection_sg_rate,
                "final_projection_valid_rate": final_projection_valid_rate,
                "final_projection_mean_min_pair": final_projection_mean_min_pair,
                "PASS": passed,
                "status": "PASS" if passed else "FAIL",
                "failure_category": None if passed else "SCHEDULE_WORSE_THAN_FINAL_PROJECTION",
                "action_needed": None if passed else "sweep rho/tau/projection timing against explicit final projection baseline",
            })
        except Exception as exc:
            rows.append(error_row("K_schedule_sweep", "all", exc, "SCHEDULE_SWEEP_ERROR"))
    df_K = dataframe_result("K_schedule_sweep", rows)
df_K


,test,graph,PASS,status,reason,failure_category,action_needed
0,K_schedule_sweep,sweep,False,SKIP,RUN_SWEEP_K=False; enable for expensive Algori...,SKIPPED_BY_FLAG,enable the corresponding RUN_* flag


## 14. Test L — operator-split versus fused CASAL update gap

**Actually tests:** Synthetic diagnostic only: estimates operator-split sensitivity under a tiny random drift, not the true fused KLDM vector-field gap.


In [55]:

rows = []
for graph_idx0, graph_id in zip(GRAPH_INDICES, GRAPH_IDS):
    try:
        g = graph_tensors(graph_idx0)
        state = _init_graph_state(
            graph_idx=graph_idx0, requested_sg=g["sg"], pos=g["pos"], l=g["l"], h=g["h"],
            lattice_transform=runner.lattice_transform, config=NONORACLE_ALGO10, template_prior=template_prior,
            oracle_reference_structure=None,
        )
        # Synthetic diagnostic only: this is not the true fused KLDM vector field.
        x_f = g["pos"]
        x_k = _l_to_k(l=g["l"], num_atoms=g["num_atoms"], lattice_transform=runner.lattice_transform)
        drift_f = 1e-3 * torch.randn_like(x_f)
        drift_k = 1e-3 * torch.randn_like(x_k)
        rho = NONORACLE_ALGO10.casal_rho_end
        tau_step = 1e-4
        r_before_f = wrap_delta(x_f - state.z_pos + state.mu_f)
        r_before_k = x_k - state.z_k + state.mu_k
        x_prior_f = torch.remainder(x_f + drift_f, 1.0)
        x_prior_k = x_k + drift_k
        r_after_f = wrap_delta(x_prior_f - state.z_pos + state.mu_f)
        r_after_k = x_prior_k - state.z_k + state.mu_k
        split_f = torch.remainder(x_prior_f - tau_step * rho * r_after_f, 1.0)
        split_k = x_prior_k - tau_step * rho * r_after_k
        fused_f = torch.remainder(x_prior_f - tau_step * rho * r_before_f, 1.0)
        fused_k = x_prior_k - tau_step * rho * r_before_k
        diff_frac = float(torch.linalg.norm(wrap_delta(split_f - fused_f).reshape(-1)).item())
        diff_k = float(torch.linalg.norm((split_k - fused_k).reshape(-1)).item())
        denom = float(torch.linalg.norm(torch.cat([drift_f.reshape(-1), drift_k.reshape(-1)])).clamp_min(1e-12).item())
        relative_diff = (diff_frac + diff_k) / denom
        passed = math.isfinite(relative_diff) and relative_diff < 1.0
        rows.append({
            "test": "L_synthetic_operator_split_gap",
            "graph": graph_id,
            "step": "synthetic_small_drift",
            "time": None,
            "diff_frac_norm": diff_frac,
            "diff_k_norm": diff_k,
            "relative_diff": relative_diff,
            "required_go_no_go": False,
            "PASS": passed,
            "status": "PASS" if passed else "FAIL",
            "failure_category": None if passed else "SYNTHETIC_OPERATOR_SPLIT_GAP_TOO_LARGE",
            "action_needed": None if passed else "record real KLDM pre/post states before claiming fused/operator-split faithfulness",
        })
    except Exception as exc:
        rows.append(error_row("L_synthetic_operator_split_gap", graph_id, exc, "SYNTHETIC_OPERATOR_SPLIT_GAP_ERROR"))
df_L = dataframe_result("L_synthetic_operator_split_gap", rows)
df_L


,test,graph,step,time,diff_frac_norm,diff_k_norm,relative_diff,required_go_no_go,PASS,status,failure_category,action_needed
0,L_synthetic_operator_split_gap,1,synthetic_small_drift,None,6.854534e-07,5.318194e-07,0.000220,False,True,PASS,None,None
1,L_synthetic_operator_split_gap,2,synthetic_small_drift,None,1.232030e-06,6.234201e-07,0.000203,False,True,PASS,None,None
2,L_synthetic_operator_split_gap,3,synthetic_small_drift,None,9.624812e-07,3.894164e-07,0.000197,False,True,PASS,None,None


## 15. Test M — final output sanity: return `z_T`, zero velocity

**Actually tests:** Checks that sampler outputs constrained z_T with near-zero velocity and deployable validation constraints.


In [56]:

if not RUN_MODEL_SAMPLING_TESTS:
    df_M = skipped_df("M_final_output_zT", "RUN_MODEL_SAMPLING_TESTS=False")
else:
    rows = []
    try:
        cfg = algorithm10_smoke_cfg()
        set_seed(SAMPLE_SEED)
        pos, v, l, h = runner._sample_batch(batch, sampler_cfg=cfg)
        source_has_zero_velocity_contract = "zeros_like" in inspect.getsource(__import__("kldmPlus.algorithm10_casal_chart", fromlist=["sample_kldm_casal_chart"]).sample_kldm_casal_chart)
        for graph_idx0, graph_id in zip(GRAPH_INDICES, GRAPH_IDS):
            target = graph_tensors(graph_idx0)
            start, end = graph_slice(graph_idx0)
            pred_structure = structure_from_graph(graph_idx0, source=(pos, l, h))
            validation = validate_requested_space_group(
                structure=pred_structure,
                requested_space_group=target["sg"],
                expected_atomic_numbers=target["h"],
                symprec=NONORACLE_ALGO10.symprec,
                angle_tolerance=NONORACLE_ALGO10.angle_tolerance,
            )
            min_pair = min_pair_distance_structure(pred_structure)
            target_volume = structure_from_graph(graph_idx0).volume
            volume_ratio = float(pred_structure.volume / max(target_volume, 1e-8))
            volume_ok = volume_ratio_ok(NONORACLE_ALGO10, volume_ratio)
            velocity_norm = float(torch.linalg.norm(v[start:end].reshape(-1)).item())
            velocity_zero = velocity_norm < 1e-12
            passed = bool(
                velocity_zero
                and validation.requested_space_group_match
                and validation.composition_match
                and min_pair >= NONORACLE_ALGO10.hard_min_distance
                and volume_ok
            )
            rows.append({
                "test": "M_final_output_zT",
                "graph": graph_id,
                "source_has_zero_velocity_contract": source_has_zero_velocity_contract,
                "velocity_norm": velocity_norm,
                "velocity_zero": velocity_zero,
                "final_sg": validation.detected_space_group,
                "requested_sg_match": validation.requested_space_group_match,
                "composition_match": validation.composition_match,
                "min_pair": min_pair,
                "volume_ratio": volume_ratio,
                "volume_ratio_ok": volume_ok,
                "n_steps": int(SAMPLING_SMOKE_N_STEPS),
                "projection_start_fraction": float(SAMPLING_SMOKE_PROJECTION_START_FRACTION),
                "projection_interval": int(SAMPLING_SMOKE_PROJECTION_INTERVAL),
                "PASS": passed,
                "status": "PASS" if passed else "FAIL",
                "failure_category": None if passed else "FINAL_OUTPUT_NOT_CONSTRAINED_ZT",
                "action_needed": None if passed else "inspect sample_kldm_casal_chart return path, zero-velocity tensor, and final validation",
            })
    except Exception as exc:
        rows.append(error_row("M_final_output_zT", "all", exc, "FINAL_OUTPUT_TEST_ERROR"))
    df_M = dataframe_result("M_final_output_zT", rows)
df_M


,test,graph,source_has_zero_velocity_contract,velocity_norm,velocity_zero,final_sg,requested_sg_match,composition_match,min_pair,volume_ratio,volume_ratio_ok,n_steps,projection_start_fraction,projection_interval,PASS,status,failure_category,action_needed
0,M_final_output_zT,1,False,0.0,True,227,True,True,1.808588,0.312681,True,60,0.75,10,True,PASS,None,None
1,M_final_output_zT,2,False,0.0,True,4,True,True,1.372910,1.288636,True,60,0.75,10,True,PASS,None,None
2,M_final_output_zT,3,False,0.0,True,194,True,True,2.721692,1.817168,True,60,0.75,10,True,PASS,None,None


## 16. Test N — CSP reconstruction using trained KLDM

**Actually tests:** Compares KLDM, explicit non-oracle final projection, and CASAL-lite with proper best-of-K aggregation.


In [57]:

def sample_with_cfg(name, cfg, *, k=1):
    all_rows = []
    for sample_idx in range(k):
        set_seed(SAMPLE_SEED + sample_idx)
        pos, _v, l, h = runner._sample_batch(batch, sampler_cfg=cfg)
        method = f"{name}@{k}#{sample_idx+1}"
        _caches[("method_source", method)] = (pos.detach().clone(), l.detach().clone(), h.detach().clone())
        all_rows.extend(evaluate_prediction_rows("N_csp_reconstruction", pos, l, h, method=method))
    return all_rows


def sample_kldm_then_project_rows(*, k=1):
    all_rows = []
    for sample_idx in range(k):
        set_seed(SAMPLE_SEED + sample_idx)
        pos, _v, l, h = runner._sample_batch(batch, sampler_cfg=facit_small)
        all_rows.extend(project_source_rows_to_chart(
            "N_csp_reconstruction",
            (pos, l, h),
            config=NONORACLE_ALGO10,
            method=f"FinalProjection@{k}#{sample_idx+1}",
        ))
    return all_rows

rows = []
if not RUN_MODEL_SAMPLING_TESTS:
    df_N = skipped_df("N_csp_reconstruction", "RUN_MODEL_SAMPLING_TESTS=False")
else:
    try:
        k_values = [max(1, int(SAMPLING_SMOKE_K))] + ([K_RECON_HEAVY] if RUN_RECONSTRUCTION_AT20 else [])
        casal_cfg = algorithm10_smoke_cfg()
        facit_small = facit_smoke_cfg()
        for k in k_values:
            rows.extend(sample_with_cfg("KLDM", facit_small, k=k))
            rows.extend(sample_kldm_then_project_rows(k=k))
            rows.extend(sample_with_cfg("Algorithm10-CASAL-lite", casal_cfg, k=k))
        df_N_raw = pd.DataFrame(rows)
        if not df_N_raw.empty:
            df_N_raw["method_base"] = df_N_raw["method"].map(method_base_from_label)
            df_N_raw["k"] = df_N_raw["method"].map(method_k_from_label)
            df_N_raw["oracle_selection_score"] = (
                (~df_N_raw["structure_match"].fillna(False).astype(bool)).astype(float) * 100.0
                + (~df_N_raw["requested_sg_match"].fillna(False).astype(bool)).astype(float) * 10.0
                + (~df_N_raw["valid"].fillna(False).astype(bool)).astype(float) * 5.0
                + df_N_raw["rmse"].fillna(10.0).astype(float)
                + np.square(np.maximum(0.0, NONORACLE_ALGO10.hard_min_distance - df_N_raw["min_pair_distance"].fillna(0.0).astype(float)))
            )
            df_N_raw["PASS"] = df_N_raw["valid"].fillna(False).astype(bool) & df_N_raw["composition_match"].fillna(False).astype(bool)
            df_N_raw["status"] = np.where(df_N_raw["PASS"], "PASS", "FAIL")
            df_N_raw["failure_category"] = np.where(df_N_raw["PASS"], None, "CSP_RECONSTRUCTION_FAILURE")
            df_N_raw["action_needed"] = np.where(df_N_raw["PASS"], None, "compare KLDM, explicit final projection, and CASAL-lite scores")
            df_N_best = (
                df_N_raw.sort_values("oracle_selection_score")
                .groupby(["method_base", "k", "graph"], as_index=False)
                .head(1)
                .reset_index(drop=True)
            )
            agg_rows = []
            for (method_base, kval), sub in df_N_best.groupby(["method_base", "k"]):
                rmse_defined = sub["rmse"].dropna()
                agg_rows.append({
                    "test": "N_csp_reconstruction_best_of_k",
                    "method_base": method_base,
                    "k": int(kval),
                    "graphs": int(sub["graph"].nunique()),
                    "validity@K": float(sub["valid"].fillna(False).astype(bool).mean()),
                    "match_rate@K": float(sub["structure_match"].fillna(False).astype(bool).mean()),
                    "requested_sg_match@K": float(sub["requested_sg_match"].fillna(False).astype(bool).mean()),
                    "composition_match@K": float(sub["composition_match"].fillna(False).astype(bool).mean()),
                    "mean_rmse@K": float(rmse_defined.mean()) if len(rmse_defined) else np.nan,
                    "mean_score@K": float(sub["oracle_selection_score"].mean()),
                    "n_steps": int(SAMPLING_SMOKE_N_STEPS),
                    "projection_start_fraction": float(SAMPLING_SMOKE_PROJECTION_START_FRACTION),
                    "projection_interval": int(SAMPLING_SMOKE_PROJECTION_INTERVAL),
                    "PASS": bool(sub["valid"].fillna(False).astype(bool).all() and sub["composition_match"].fillna(False).astype(bool).all()),
                })
            df_N_best_summary_raw = pd.DataFrame(agg_rows)
            df_N_best_summary_raw["status"] = np.where(df_N_best_summary_raw["PASS"], "PASS", "FAIL")
            df_N_best_summary_raw["failure_category"] = np.where(df_N_best_summary_raw["PASS"], None, "BEST_OF_K_RECONSTRUCTION_FAILURE")
            df_N_best_summary_raw["action_needed"] = np.where(df_N_best_summary_raw["PASS"], None, "inspect per-graph best rows and explicit projection baseline")
            df_N_best_summary = dataframe_result("N_csp_reconstruction_best_of_k", df_N_best_summary_raw.to_dict("records"))
        else:
            df_N_best = pd.DataFrame()
            df_N_best_summary = pd.DataFrame()
        df_N = dataframe_result("N_csp_reconstruction", df_N_raw.to_dict("records"))
        display(df_N)
        display(df_N_best)
        df_N_best_summary
    except Exception as exc:
        df_N = dataframe_result("N_csp_reconstruction", [error_row("N_csp_reconstruction", "all", exc, "CSP_RECONSTRUCTION_ERROR")])
df_N


,test,method,method_base,k,graph,valid,composition_match,requested_sg_match,detected_sg,structure_match,...,projection_objective,projection_coord_loss,projection_lattice_k_loss,projection_min_pair,projection_volume_ratio,projection_template_labels,projection_tau,oracle_selection_score,failure_category,action_needed
0,N_csp_reconstruction,KLDM@1#1,KLDM,1,1,True,True,False,2,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000042e+01,None,None
1,N_csp_reconstruction,KLDM@1#1,KLDM,1,2,True,True,False,7,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.200000e+02,None,None
2,N_csp_reconstruction,KLDM@1#1,KLDM,1,3,True,True,False,1,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.200000e+02,None,None
3,N_csp_reconstruction,FinalProjection@1#1,FinalProjection,1,1,True,True,True,227,True,...,0.000437,7.022027e-09,0.000027,2.655575,1.000000,"70@8b,77@16c","(0.071, 0.1081, 0.7266)",6.119292e-16,None,None
4,N_csp_reconstruction,FinalProjection@1#1,FinalProjection,1,2,True,True,True,4,False,...,0.007767,8.142124e-03,0.000033,1.422012,1.000001,"26@2a,26@2a,26@2a,27@2a,27@2a,27@2a,5@2a,5@2a","(0.6532, 0.7736, 0.7552)",1.100000e+02,None,None
5,N_csp_reconstruction,FinalProjection@1#1,FinalProjection,1,3,True,True,True,194,False,...,0.001759,1.281430e-03,0.000063,1.957835,1.000000,"72@2c,22@4f,22@2d","(0.087, 0.4646, 0.1285)",1.100000e+02,None,None
6,N_csp_reconstruction,Algorithm10-CASAL-lite@1#1,Algorithm10-CASAL-lite,1,1,True,True,True,227,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.762987e-16,None,None
7,N_csp_reconstruction,Algorithm10-CASAL-lite@1#1,Algorithm10-CASAL-lite,1,2,True,True,True,4,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.100000e+02,None,None
8,N_csp_reconstruction,Algorithm10-CASAL-lite@1#1,Algorithm10-CASAL-lite,1,3,True,True,True,194,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.100000e+02,None,None


,test,method,method_base,k,graph,valid,composition_match,requested_sg_match,detected_sg,structure_match,...,projection_objective,projection_coord_loss,projection_lattice_k_loss,projection_min_pair,projection_volume_ratio,projection_template_labels,projection_tau,oracle_selection_score,failure_category,action_needed
0,N_csp_reconstruction,Algorithm10-CASAL-lite@1#1,Algorithm10-CASAL-lite,1,1,True,True,True,227,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.762987e-16,None,None
1,N_csp_reconstruction,FinalProjection@1#1,FinalProjection,1,1,True,True,True,227,True,...,0.000437,7.022027e-09,0.000027,2.655575,1.000000,"70@8b,77@16c","(0.071, 0.1081, 0.7266)",6.119292e-16,None,None
2,N_csp_reconstruction,KLDM@1#1,KLDM,1,1,True,True,False,2,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000042e+01,None,None
3,N_csp_reconstruction,FinalProjection@1#1,FinalProjection,1,2,True,True,True,4,False,...,0.007767,8.142124e-03,0.000033,1.422012,1.000001,"26@2a,26@2a,26@2a,27@2a,27@2a,27@2a,5@2a,5@2a","(0.6532, 0.7736, 0.7552)",1.100000e+02,None,None
4,N_csp_reconstruction,FinalProjection@1#1,FinalProjection,1,3,True,True,True,194,False,...,0.001759,1.281430e-03,0.000063,1.957835,1.000000,"72@2c,22@4f,22@2d","(0.087, 0.4646, 0.1285)",1.100000e+02,None,None
5,N_csp_reconstruction,Algorithm10-CASAL-lite@1#1,Algorithm10-CASAL-lite,1,2,True,True,True,4,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.100000e+02,None,None
6,N_csp_reconstruction,Algorithm10-CASAL-lite@1#1,Algorithm10-CASAL-lite,1,3,True,True,True,194,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.100000e+02,None,None
7,N_csp_reconstruction,KLDM@1#1,KLDM,1,2,True,True,False,7,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.200000e+02,None,None
8,N_csp_reconstruction,KLDM@1#1,KLDM,1,3,True,True,False,1,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.200000e+02,None,None


,test,method,method_base,k,graph,valid,composition_match,requested_sg_match,detected_sg,structure_match,...,projection_objective,projection_coord_loss,projection_lattice_k_loss,projection_min_pair,projection_volume_ratio,projection_template_labels,projection_tau,oracle_selection_score,failure_category,action_needed
0,N_csp_reconstruction,KLDM@1#1,KLDM,1,1,True,True,False,2,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000042e+01,None,None
1,N_csp_reconstruction,KLDM@1#1,KLDM,1,2,True,True,False,7,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.200000e+02,None,None
2,N_csp_reconstruction,KLDM@1#1,KLDM,1,3,True,True,False,1,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.200000e+02,None,None
3,N_csp_reconstruction,FinalProjection@1#1,FinalProjection,1,1,True,True,True,227,True,...,0.000437,7.022027e-09,0.000027,2.655575,1.000000,"70@8b,77@16c","(0.071, 0.1081, 0.7266)",6.119292e-16,None,None
4,N_csp_reconstruction,FinalProjection@1#1,FinalProjection,1,2,True,True,True,4,False,...,0.007767,8.142124e-03,0.000033,1.422012,1.000001,"26@2a,26@2a,26@2a,27@2a,27@2a,27@2a,5@2a,5@2a","(0.6532, 0.7736, 0.7552)",1.100000e+02,None,None
5,N_csp_reconstruction,FinalProjection@1#1,FinalProjection,1,3,True,True,True,194,False,...,0.001759,1.281430e-03,0.000063,1.957835,1.000000,"72@2c,22@4f,22@2d","(0.087, 0.4646, 0.1285)",1.100000e+02,None,None
6,N_csp_reconstruction,Algorithm10-CASAL-lite@1#1,Algorithm10-CASAL-lite,1,1,True,True,True,227,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.762987e-16,None,None
7,N_csp_reconstruction,Algorithm10-CASAL-lite@1#1,Algorithm10-CASAL-lite,1,2,True,True,True,4,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.100000e+02,None,None
8,N_csp_reconstruction,Algorithm10-CASAL-lite@1#1,Algorithm10-CASAL-lite,1,3,True,True,True,194,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.100000e+02,None,None


## 17. Test O — CASAL-lite ablations

**Actually tests:** Separates projection feasibility from useful split dynamics. It runs fixed-`x` projection/dual loops, repeats the loop with fixed `(W, tau, pi)`, and compares CASAL-lite against final-only projection. These are diagnostics: failure means CASAL-lite is not yet adding guidance, not that the chart projection is broken.


In [58]:


def xz_residual_stats(x_pos, x_k, projection):
    coord = torch.linalg.norm(wrap_delta(x_pos - projection.pos).reshape(-1))
    kval = torch.linalg.norm((x_k - projection.k).reshape(-1))
    total = torch.sqrt(coord.square() + kval.square())
    return {
        "coord": float(coord.detach().item()),
        "k": float(kval.detach().item()),
        "total": float(total.detach().item()),
    }


def xz_residual_norm(x_pos, x_k, projection):
    return xz_residual_stats(x_pos, x_k, projection)["total"]


def _dataclass_replace_supported(obj, **updates):
    try:
        names = {field.name for field in dataclasses.fields(obj)}
    except TypeError:
        return obj
    supported = {key: value for key, value in updates.items() if key in names}
    if not supported:
        return obj
    return dataclasses.replace(obj, **supported)


def projection_arg_l_from_k(k, num_atoms):
    return _k_to_l(k=k.reshape(-1), num_atoms=int(num_atoms), lattice_transform=runner.lattice_transform).reshape(-1)


def fixed_template_projection_for_arg(*, graph_idx0, template_state, tau0, arg_pos, arg_k, h, freeze_tau, use_fixed_assignment, fixed_assignment=None):
    arg_l = projection_arg_l_from_k(arg_k, int(arg_pos.shape[0])).to(device=arg_pos.device, dtype=arg_pos.dtype)
    arg_cell = _decode_lattice_matrix(
        l=arg_l.reshape(-1),
        num_atoms=int(arg_pos.shape[0]),
        lattice_transform=runner.lattice_transform,
    ).to(device=arg_pos.device, dtype=arg_pos.dtype)
    chart_frac, chart_h, chart_cell, chart_k = _build_chart_target(
        frac_coords=arg_pos,
        atomic_numbers=h,
        cell_matrix=arg_cell,
        requested_sg=graph_tensors(graph_idx0)["sg"],
        symprec=NONORACLE_ALGO10.symprec,
        angle_tolerance=NONORACLE_ALGO10.angle_tolerance,
    )
    state = template_state
    if fixed_assignment is not None:
        state = _dataclass_replace_supported(state, fixed_target_assignment=fixed_assignment.detach().clone())
    ssvd_cfg = _dataclass_replace_supported(
        NONORACLE_ALGO10.ssvd_projection_config(),
        freeze_tau=bool(freeze_tau),
        use_fixed_assignment=bool(use_fixed_assignment),
    )
    result = fixed_template_ssvd_project(
        template_state=state,
        y_f=chart_frac,
        y_k=chart_k,
        y_h=chart_h.to(device=arg_pos.device, dtype=torch.long),
        tau0=tau0,
        metric=NONORACLE_ALGO10.projection_metric(),
        config=ssvd_cfg,
    )
    projection = _materialize_projection(
        graph_idx=graph_idx0,
        projection=result,
        target_pos=arg_pos,
        target_h=h,
        target_cell=arg_cell,
        requested_sg=graph_tensors(graph_idx0)["sg"],
        lattice_transform=runner.lattice_transform,
        config=NONORACLE_ALGO10,
    )
    return projection, result


def _metric_for_mode(stats, mode):
    if mode == "coord_only_dual":
        return stats["coord"]
    if mode == "lattice_only_dual":
        return stats["k"]
    return stats["total"]


def _run_dual_ablation_sequence(*, graph_idx0, graph_id, mode, eta, x_pos, x_k, h, init_projection, init_fixed_result=None):
    mu_f = torch.zeros_like(x_pos)
    mu_k = torch.zeros_like(x_k)
    projection = init_projection
    initial_stats = xz_residual_stats(x_pos, x_k, projection)
    residuals_total = [initial_stats["total"]]
    residuals_coord = [initial_stats["coord"]]
    residuals_k = [initial_stats["k"]]
    metric_initial = _metric_for_mode(initial_stats, mode)
    metric_values = [metric_initial]
    template_sequence = [projection.template_labels]
    tau_sequence = [tuple(round(float(v), 4) for v in projection.tau.reshape(-1).detach().cpu().tolist())]
    fixed_tau = None
    fixed_assignment = None
    fixed_state = None
    freeze_supported = "freeze_tau" in {field.name for field in dataclasses.fields(NONORACLE_ALGO10.ssvd_projection_config())}
    fixed_assignment_supported = False

    if mode == "fixed_W_tau_pi_dual":
        if init_fixed_result is None:
            init_fixed_projection, init_fixed_result = fixed_template_projection_for_arg(
                graph_idx0=graph_idx0,
                template_state=init_projection.state,
                tau0=init_projection.tau,
                arg_pos=x_pos,
                arg_k=x_k,
                h=h,
                freeze_tau=True,
                use_fixed_assignment=False,
            )
            projection = init_fixed_projection
            initial_stats = xz_residual_stats(x_pos, x_k, projection)
            metric_initial = _metric_for_mode(initial_stats, mode)
            residuals_total = [initial_stats["total"]]
            residuals_coord = [initial_stats["coord"]]
            residuals_k = [initial_stats["k"]]
            metric_values = [metric_initial]
        fixed_tau = init_fixed_result.tau.detach().clone()
        fixed_assignment = init_fixed_result.assignment.detach().clone()
        fixed_state = _dataclass_replace_supported(init_fixed_result.state, fixed_target_assignment=fixed_assignment)
        fixed_assignment_supported = fixed_state is not init_fixed_result.state or hasattr(fixed_state, "fixed_target_assignment")

    for _step in range(int(CASAL_ABLATION_STEPS)):
        if mode == "lattice_only_dual":
            arg_pos = x_pos
        else:
            arg_pos = torch.remainder(x_pos + mu_f, 1.0)
        if mode == "coord_only_dual":
            arg_k = x_k
        else:
            arg_k = x_k + mu_k

        if mode == "fixed_W_tau_pi_dual":
            projection, result = fixed_template_projection_for_arg(
                graph_idx0=graph_idx0,
                template_state=fixed_state,
                tau0=fixed_tau,
                arg_pos=arg_pos,
                arg_k=arg_k,
                h=h,
                freeze_tau=True,
                use_fixed_assignment=True,
                fixed_assignment=fixed_assignment,
            )
            fixed_state = _dataclass_replace_supported(result.state, fixed_target_assignment=fixed_assignment)
        else:
            arg_l = projection_arg_l_from_k(arg_k, int(x_pos.shape[0])).to(device=x_pos.device, dtype=x_pos.dtype)
            projection = _project_graph_to_chart(
                graph_idx=graph_idx0,
                requested_sg=graph_tensors(graph_idx0)["sg"],
                target_pos=arg_pos,
                target_l=arg_l,
                target_h=h,
                lattice_transform=runner.lattice_transform,
                config=NONORACLE_ALGO10,
                template_prior=template_prior,
                oracle_reference_structure=None,
            )

        residual_f = wrap_delta(x_pos - projection.pos)
        residual_k = x_k - projection.k
        if mode != "lattice_only_dual":
            mu_f = mu_f + float(eta) * residual_f
        if mode != "coord_only_dual":
            mu_k = mu_k + float(eta) * residual_k
        clip = float(NONORACLE_ALGO10.casal_mu_clip)
        if clip > 0:
            mu_f = mu_f.clamp(min=-clip, max=clip)
            mu_k = mu_k.clamp(min=-clip, max=clip)

        stats = xz_residual_stats(x_pos, x_k, projection)
        residuals_total.append(stats["total"])
        residuals_coord.append(stats["coord"])
        residuals_k.append(stats["k"])
        metric_values.append(_metric_for_mode(stats, mode))
        template_sequence.append(projection.template_labels)
        tau_sequence.append(tuple(round(float(v), 4) for v in projection.tau.reshape(-1).detach().cpu().tolist()))

    metric_final = metric_values[-1]
    metric_best = min(metric_values)
    final_reduced = bool(metric_final <= metric_initial + CASAL_ABLATION_IMPROVE_TOL)
    best_reduced = bool(metric_best <= metric_initial + CASAL_ABLATION_IMPROVE_TOL)
    return {
        "test": "O_casal_lite_ablations_detail",
        "graph": graph_id,
        "mode": mode,
        "eta_mu": float(eta),
        "residual_initial_total": residuals_total[0],
        "residual_final_total": residuals_total[-1],
        "residual_best_total": min(residuals_total),
        "residual_initial_coord": residuals_coord[0],
        "residual_final_coord": residuals_coord[-1],
        "residual_best_coord": min(residuals_coord),
        "residual_initial_k": residuals_k[0],
        "residual_final_k": residuals_k[-1],
        "residual_best_k": min(residuals_k),
        "metric_initial": metric_initial,
        "metric_final": metric_final,
        "metric_best": metric_best,
        "metric_final_reduced": final_reduced,
        "metric_best_reduced": best_reduced,
        "template_switch_count": max(0, len(set(template_sequence)) - 1),
        "tau_switch_count": max(0, len(set(tau_sequence)) - 1),
        "mu_f_norm": float(torch.linalg.norm(mu_f.reshape(-1)).detach().item()),
        "mu_k_norm": float(torch.linalg.norm(mu_k.reshape(-1)).detach().item()),
        "freeze_tau_supported": bool(freeze_supported),
        "fixed_assignment_supported": bool(fixed_assignment_supported),
        "PASS": final_reduced,
        "status": "PASS" if final_reduced else "FAIL",
    }


detail_rows = []
summary_rows = []
if not RUN_MODEL_SAMPLING_TESTS:
    df_O = skipped_df("O_casal_lite_ablations", "RUN_MODEL_SAMPLING_TESTS=False")
else:
    try:
        kldm_source = get_kldm_sample_once()
        for graph_idx0, graph_id in zip(GRAPH_INDICES, GRAPH_IDS):
            try:
                g = graph_tensors(graph_idx0, source=kldm_source)
                x_pos = g["pos"].to(device=runner.device, dtype=dtype)
                h = g["h"].to(device=runner.device, dtype=torch.long)
                x_k = _l_to_k(l=g["l"], num_atoms=g["num_atoms"], lattice_transform=runner.lattice_transform).to(device=runner.device, dtype=dtype)
                init_state = _init_graph_state(
                    graph_idx=graph_idx0,
                    requested_sg=g["sg"],
                    pos=x_pos,
                    l=g["l"],
                    h=h,
                    lattice_transform=runner.lattice_transform,
                    config=NONORACLE_ALGO10,
                    template_prior=template_prior,
                    oracle_reference_structure=None,
                )
                init_fixed_result = None
                if "fixed_W_tau_pi_dual" in CASAL_ABLATION_MODES:
                    try:
                        _, init_fixed_result = fixed_template_projection_for_arg(
                            graph_idx0=graph_idx0,
                            template_state=init_state.projection.state,
                            tau0=init_state.projection.tau,
                            arg_pos=x_pos,
                            arg_k=x_k,
                            h=h,
                            freeze_tau=True,
                            use_fixed_assignment=False,
                        )
                    except Exception:
                        init_fixed_result = None
                for mode in CASAL_ABLATION_MODES:
                    mode_rows = []
                    for eta in CASAL_ABLATION_ETA_SWEEP:
                        try:
                            row = _run_dual_ablation_sequence(
                                graph_idx0=graph_idx0,
                                graph_id=graph_id,
                                mode=mode,
                                eta=float(eta),
                                x_pos=x_pos,
                                x_k=x_k,
                                h=h,
                                init_projection=init_state.projection,
                                init_fixed_result=init_fixed_result,
                            )
                        except Exception as exc:
                            row = error_row("O_casal_lite_ablations_detail", graph_id, exc, "CASAL_ABLATION_ERROR")
                            row.update({"mode": mode, "eta_mu": float(eta), "PASS": False, "status": "ERROR"})
                        detail_rows.append(row)
                        mode_rows.append(row)
                    ok_rows = [r for r in mode_rows if r.get("status") in {"PASS", "FAIL"}]
                    if not ok_rows:
                        summary_rows.append(error_row("O_casal_lite_ablations", graph_id, RuntimeError(f"no successful rows for {mode}"), "CASAL_ABLATION_ERROR") | {"mode": mode})
                        continue
                    best_final = min(ok_rows, key=lambda r: float(r.get("metric_final", float("inf"))))
                    best_any = min(ok_rows, key=lambda r: float(r.get("metric_best", float("inf"))))
                    passed = any(bool(r.get("metric_final_reduced")) for r in ok_rows)
                    summary_rows.append({
                        "test": "O_casal_lite_ablations",
                        "graph": graph_id,
                        "mode": mode,
                        "best_eta_final": float(best_final.get("eta_mu", np.nan)),
                        "best_eta_any": float(best_any.get("eta_mu", np.nan)),
                        "metric_initial": float(best_final.get("metric_initial", np.nan)),
                        "best_metric_final": float(best_final.get("metric_final", np.nan)),
                        "best_metric_any": float(best_any.get("metric_best", np.nan)),
                        "final_reduction_available": bool(passed),
                        "best_reduction_available": any(bool(r.get("metric_best_reduced")) for r in ok_rows),
                        "min_template_switch_count": int(min(float(r.get("template_switch_count", 9999)) for r in ok_rows)),
                        "min_tau_switch_count": int(min(float(r.get("tau_switch_count", 9999)) for r in ok_rows)),
                        "PASS": bool(passed),
                        "status": "PASS" if passed else "FAIL",
                        "failure_category": None if passed else f"{mode.upper()}_NOT_STABILIZING",
                        "action_needed": None if passed else "reduce eta/rho, lock gauge, or separate coordinate and lattice coupling before testing fused CASAL",
                    })
            except Exception as exc:
                summary_rows.append(error_row("O_casal_lite_ablations", graph_id, exc, "CASAL_ABLATION_ERROR"))
        df_O_detail = pd.DataFrame(detail_rows)
        result_tables["O_casal_lite_ablations_detail"] = df_O_detail
        df_O = dataframe_result("O_casal_lite_ablations", summary_rows)
    except Exception as exc:
        df_O = dataframe_result("O_casal_lite_ablations", [error_row("O_casal_lite_ablations", "all", exc, "KLDM_SAMPLE_ERROR")])
print("Detailed eta-sweep rows are stored in result_tables['O_casal_lite_ablations_detail']")
df_O


Detailed eta-sweep rows are stored in result_tables['O_casal_lite_ablations_detail']


,test,graph,mode,best_eta_final,best_eta_any,metric_initial,best_metric_final,best_metric_any,final_reduction_available,best_reduction_available,min_template_switch_count,min_tau_switch_count,PASS,status,failure_category,action_needed
0,O_casal_lite_ablations,1,full_projection_dual,0.025,0.010,1.210446,0.786695,0.786695,True,True,1,5,True,PASS,None,None
1,O_casal_lite_ablations,1,fixed_W_tau_pi_dual,0.050,0.010,1.210446,1.210485,1.210446,False,True,0,5,False,FAIL,FIXED_W_TAU_PI_DUAL_NOT_STABILIZING,"reduce eta/rho, lock gauge, or separate coordi..."
2,O_casal_lite_ablations,1,coord_only_dual,0.100,0.100,1.210407,0.804048,0.804048,True,True,0,5,True,PASS,None,None
3,O_casal_lite_ablations,1,lattice_only_dual,0.250,0.050,0.009741,0.009741,0.009741,True,True,0,0,True,PASS,None,None
4,O_casal_lite_ablations,2,full_projection_dual,0.100,0.100,1.055491,1.207961,0.754124,False,True,0,6,False,FAIL,FULL_PROJECTION_DUAL_NOT_STABILIZING,"reduce eta/rho, lock gauge, or separate coordi..."
5,O_casal_lite_ablations,2,fixed_W_tau_pi_dual,0.010,0.010,1.055491,1.079367,1.055491,False,True,0,5,False,FAIL,FIXED_W_TAU_PI_DUAL_NOT_STABILIZING,"reduce eta/rho, lock gauge, or separate coordi..."
6,O_casal_lite_ablations,2,coord_only_dual,0.025,0.025,1.055398,1.111921,0.674398,False,True,0,6,False,FAIL,COORD_ONLY_DUAL_NOT_STABILIZING,"reduce eta/rho, lock gauge, or separate coordi..."
7,O_casal_lite_ablations,2,lattice_only_dual,0.010,0.010,0.013992,0.013992,0.013992,True,True,0,6,True,PASS,None,None
8,O_casal_lite_ablations,3,full_projection_dual,0.010,0.250,1.219640,1.159451,1.156765,True,True,1,5,True,PASS,None,None
9,O_casal_lite_ablations,3,fixed_W_tau_pi_dual,0.010,0.010,1.219640,1.222263,1.219640,False,True,0,5,False,FAIL,FIXED_W_TAU_PI_DUAL_NOT_STABILIZING,"reduce eta/rho, lock gauge, or separate coordi..."


## 18. Test P — final projection versus CASAL-lite comparison

**Actually tests:** Checks whether CASAL-lite improves any observable over final-only projection on the same smoke sample. This is a practical diagnostic for whether split coupling adds value beyond `KLDM -> P_C(x_T)`.


In [59]:


rows = []
if not RUN_MODEL_SAMPLING_TESTS or "N_csp_reconstruction" not in result_tables:
    df_P = skipped_df("P_final_projection_vs_casal", "RUN_MODEL_SAMPLING_TESTS=False or N not available")
else:
    df_raw = result_tables["N_csp_reconstruction"].copy()
    for graph_id in GRAPH_IDS:
        final_rows = df_raw[(df_raw.get("graph") == graph_id) & (df_raw.get("method_base") == "FinalProjection")]
        casal_rows = df_raw[(df_raw.get("graph") == graph_id) & (df_raw.get("method_base") == "Algorithm10-CASAL-lite")]
        if final_rows.empty or casal_rows.empty:
            rows.append(error_row("P_final_projection_vs_casal", graph_id, RuntimeError("missing comparison rows"), "COMPARISON_ROWS_MISSING"))
            continue
        final = final_rows.iloc[0]
        casal = casal_rows.iloc[0]
        final_rmse = final.get("rmse", np.nan)
        casal_rmse = casal.get("rmse", np.nan)
        final_min_pair = final.get("min_pair_distance", np.nan)
        casal_min_pair = casal.get("min_pair_distance", np.nan)
        final_volume = final.get("volume", np.nan)
        casal_volume = casal.get("volume", np.nan)
        target_volume = target_volume_for_graph(GRAPH_INDICES[GRAPH_IDS.index(graph_id)])
        final_volume_error = abs(float(final_volume) / max(target_volume, 1.0e-8) - 1.0) if pd.notna(final_volume) else np.nan
        casal_volume_error = abs(float(casal_volume) / max(target_volume, 1.0e-8) - 1.0) if pd.notna(casal_volume) else np.nan
        rmse_improved = bool(pd.notna(final_rmse) and pd.notna(casal_rmse) and float(casal_rmse) <= float(final_rmse) + 1.0e-8)
        match_improved = bool((not bool(final.get("structure_match"))) and bool(casal.get("structure_match")))
        min_pair_improved = bool(pd.notna(final_min_pair) and pd.notna(casal_min_pair) and float(casal_min_pair) >= float(final_min_pair) - 1.0e-8)
        volume_improved = bool(pd.notna(final_volume_error) and pd.notna(casal_volume_error) and float(casal_volume_error) <= float(final_volume_error) + 1.0e-8)
        sg_not_worse = bool(bool(casal.get("requested_sg_match")) >= bool(final.get("requested_sg_match")))
        valid_not_worse = bool(bool(casal.get("valid")) >= bool(final.get("valid")))
        improves_any = bool(match_improved or rmse_improved or min_pair_improved or volume_improved)
        passed = bool(sg_not_worse and valid_not_worse and improves_any)
        rows.append({
            "test": "P_final_projection_vs_casal",
            "graph": graph_id,
            "final_match": bool(final.get("structure_match")),
            "casal_match": bool(casal.get("structure_match")),
            "match_improved": match_improved,
            "final_rmse": final_rmse,
            "casal_rmse": casal_rmse,
            "rmse_improved": rmse_improved,
            "final_min_pair": final_min_pair,
            "casal_min_pair": casal_min_pair,
            "min_pair_improved": min_pair_improved,
            "final_volume_error": final_volume_error,
            "casal_volume_error": casal_volume_error,
            "volume_improved": volume_improved,
            "sg_not_worse": sg_not_worse,
            "valid_not_worse": valid_not_worse,
            "casal_improves_any_observable": improves_any,
            "PASS": passed,
            "status": "PASS" if passed else "FAIL",
            "failure_category": None if passed else "CASAL_LITE_NOT_BETTER_THAN_FINAL_PROJECTION",
            "action_needed": None if passed else "run fixed-x/fixed-chart ablations and tune rho/tau/projection timing before full CASAL",
        })
    df_P = dataframe_result("P_final_projection_vs_casal", rows)
df_P


,test,graph,final_match,casal_match,match_improved,final_rmse,casal_rmse,rmse_improved,final_min_pair,casal_min_pair,...,final_volume_error,casal_volume_error,volume_improved,sg_not_worse,valid_not_worse,casal_improves_any_observable,PASS,status,failure_category,action_needed
0,P_final_projection_vs_casal,1,True,True,False,6.119292e-16,4.762987e-16,True,2.655575,1.808588,...,0.010175,0.687319,False,True,True,True,True,PASS,None,None
1,P_final_projection_vs_casal,2,False,False,False,NaN,NaN,False,1.422012,1.372910,...,0.028746,0.288636,False,True,True,False,False,FAIL,CASAL_LITE_NOT_BETTER_THAN_FINAL_PROJECTION,run fixed-x/fixed-chart ablations and tune rho...
2,P_final_projection_vs_casal,3,False,False,False,NaN,NaN,False,1.957834,2.721692,...,0.209763,0.817167,False,True,True,True,True,PASS,None,None


## 19. Summary decision table

**Actually tests:** Aggregates pass/fail rows across all tests.


In [60]:
summary_df = pd.DataFrame(summary_rows)
summary_df


,test,graph,mode,best_eta_final,best_eta_any,metric_initial,best_metric_final,best_metric_any,final_reduction_available,best_reduction_available,...,status,failure_category,action_needed,test_name,graphs_tested,pass_count,fail_count,skip_count,error_count,worst_graph
0,O_casal_lite_ablations,1.0,full_projection_dual,0.025,0.010,1.210446,0.786695,0.786695,True,True,...,PASS,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,O_casal_lite_ablations,1.0,fixed_W_tau_pi_dual,0.050,0.010,1.210446,1.210485,1.210446,False,True,...,FAIL,FIXED_W_TAU_PI_DUAL_NOT_STABILIZING,"reduce eta/rho, lock gauge, or separate coordi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,O_casal_lite_ablations,1.0,coord_only_dual,0.100,0.100,1.210407,0.804048,0.804048,True,True,...,PASS,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,O_casal_lite_ablations,1.0,lattice_only_dual,0.250,0.050,0.009741,0.009741,0.009741,True,True,...,PASS,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,O_casal_lite_ablations,2.0,full_projection_dual,0.100,0.100,1.055491,1.207961,0.754124,False,True,...,FAIL,FULL_PROJECTION_DUAL_NOT_STABILIZING,"reduce eta/rho, lock gauge, or separate coordi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,O_casal_lite_ablations,2.0,fixed_W_tau_pi_dual,0.010,0.010,1.055491,1.079367,1.055491,False,True,...,FAIL,FIXED_W_TAU_PI_DUAL_NOT_STABILIZING,"reduce eta/rho, lock gauge, or separate coordi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,O_casal_lite_ablations,2.0,coord_only_dual,0.025,0.025,1.055398,1.111921,0.674398,False,True,...,FAIL,COORD_ONLY_DUAL_NOT_STABILIZING,"reduce eta/rho, lock gauge, or separate coordi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,O_casal_lite_ablations,2.0,lattice_only_dual,0.010,0.010,0.013992,0.013992,0.013992,True,True,...,PASS,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,O_casal_lite_ablations,3.0,full_projection_dual,0.010,0.250,1.219640,1.159451,1.156765,True,True,...,PASS,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,O_casal_lite_ablations,3.0,fixed_W_tau_pi_dual,0.010,0.010,1.219640,1.222263,1.219640,False,True,...,FAIL,FIXED_W_TAU_PI_DUAL_NOT_STABILIZING,"reduce eta/rho, lock gauge, or separate coordi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [61]:

# Go / no-go summary from the tutorial.
projection_required_tests = [
    "A_data_extraction",
    "B_pyxtal_extraction",
    "C_template_enumeration",
    "D_materialization_identity",
    "E_k_basis_lattice",
    "F_fixed_template_ssvd",
    "H_full_projection",
]
model_sampling_required_tests = [
    "J_one_step_casal_lite",
    "M_final_output_zT",
    "N_csp_reconstruction_best_of_k" if "N_csp_reconstruction_best_of_k" in result_tables else "N_csp_reconstruction",
]
strongly_recommended = [
    "G_wrong_template_stress",
    "I_casal_split_state",
]
diagnostic_only = [
    "K_schedule_sweep",
    "L_synthetic_operator_split_gap",
    "O_casal_lite_ablations",
    "P_final_projection_vs_casal",
]

def table_passed(name):
    df = result_tables.get(name)
    return bool(df is not None and len(df) > 0 and df["PASS"].fillna(False).astype(bool).all())

def table_status(name):
    df = result_tables.get(name)
    if df is None or len(df) == 0:
        return "NOT_RUN"
    statuses = set(str(v) for v in df.get("status", pd.Series([], dtype=str)).dropna().tolist())
    if statuses == {"SKIP"}:
        return "SKIPPED"
    return "PASS" if table_passed(name) else "FAIL"

decision_rows = []
for name in projection_required_tests:
    decision_rows.append({"tier": "projection_required", "test_name": name, "status": table_status(name), "passed": table_passed(name)})
for name in model_sampling_required_tests:
    decision_rows.append({"tier": "model_sampling_required", "test_name": name, "status": table_status(name), "passed": table_passed(name)})
for name in strongly_recommended:
    decision_rows.append({"tier": "strongly_recommended", "test_name": name, "status": table_status(name), "passed": table_passed(name)})
for name in diagnostic_only:
    if name in result_tables:
        decision_rows.append({"tier": "diagnostic_only", "test_name": name, "status": table_status(name), "passed": table_passed(name)})

decision_df = pd.DataFrame(decision_rows)
GO_FOR_ALGORITHM10_SAMPLING = bool(decision_df[decision_df["tier"] == "projection_required"]["passed"].all())
MODEL_SAMPLING_TESTS_RAN = bool(RUN_MODEL_SAMPLING_TESTS)
MODEL_SAMPLING_TESTS_PASSED = bool(
    MODEL_SAMPLING_TESTS_RAN
    and decision_df[decision_df["tier"] == "model_sampling_required"]["passed"].all()
)
GO_FOR_ALGORITHM10_TRUST = bool(GO_FOR_ALGORITHM10_SAMPLING and MODEL_SAMPLING_TESTS_PASSED)

graph3_k_residual = np.nan
if "I_casal_split_state" in result_tables and "xz_k_residual_norm" in result_tables["I_casal_split_state"].columns:
    row = result_tables["I_casal_split_state"][result_tables["I_casal_split_state"].get("graph") == 3]
    if not row.empty:
        graph3_k_residual = float(row.iloc[0]["xz_k_residual_norm"])

interpretation_rows = [
    {
        "claim": "Projection/chart stack",
        "status": "PASS" if GO_FOR_ALGORITHM10_SAMPLING else "FAIL",
        "meaning": "Ground-truth chart projection is feasible; this green-lights Algorithm10 sampling experiments.",
    },
    {
        "claim": "CASAL-lite state machinery",
        "status": table_status("I_casal_split_state"),
        "meaning": "Shape/finite split-state smoke test only; not evidence of improved generation.",
    },
    {
        "claim": "Full KLDM + CASAL-lite generation",
        "status": "PASS" if MODEL_SAMPLING_TESTS_PASSED else ("NOT_TESTED" if not MODEL_SAMPLING_TESTS_RAN else "FAIL"),
        "meaning": "J/M/N test output validity and reconstruction. O/P separately test whether CASAL-lite adds useful guidance beyond final projection.",
    },
    {
        "claim": "CASAL-lite adds guidance",
        "status": table_status("P_final_projection_vs_casal"),
        "meaning": "PASS only if CASAL-lite improves at least one observable over final-only projection without hurting validity/SG.",
    },
    {
        "claim": "Graph 3 lattice coupling risk",
        "status": "WATCH" if math.isfinite(graph3_k_residual) and graph3_k_residual > GRAPH3_K_RESIDUAL_WARN else "OK",
        "meaning": f"graph3 ||x_k-z_k||={graph3_k_residual:.4f}; watch mu_k, volume changes, and projection jumps during sampling.",
    },
]
interpretation_df = pd.DataFrame(interpretation_rows)
print(f"GO_FOR_ALGORITHM10_SAMPLING={GO_FOR_ALGORITHM10_SAMPLING}")
print(f"MODEL_SAMPLING_TESTS_RAN={MODEL_SAMPLING_TESTS_RAN}")
print(f"MODEL_SAMPLING_TESTS_PASSED={MODEL_SAMPLING_TESTS_PASSED}")
print(f"GO_FOR_ALGORITHM10_TRUST={GO_FOR_ALGORITHM10_TRUST}")
display(interpretation_df)
decision_df


GO_FOR_ALGORITHM10_SAMPLING=True
MODEL_SAMPLING_TESTS_RAN=True
MODEL_SAMPLING_TESTS_PASSED=False
GO_FOR_ALGORITHM10_TRUST=False


,claim,status,meaning
0,Projection/chart stack,PASS,Ground-truth chart projection is feasible; thi...
1,CASAL-lite state machinery,PASS,Shape/finite split-state smoke test only; not ...
2,Full KLDM + CASAL-lite generation,FAIL,J/M/N test output validity and reconstruction....
3,CASAL-lite adds guidance,FAIL,PASS only if CASAL-lite improves at least one ...
4,Graph 3 lattice coupling risk,WATCH,"graph3 ||x_k-z_k||=0.4057; watch mu_k, volume ..."


,tier,test_name,status,passed
0,projection_required,A_data_extraction,PASS,True
1,projection_required,B_pyxtal_extraction,PASS,True
2,projection_required,C_template_enumeration,PASS,True
3,projection_required,D_materialization_identity,PASS,True
4,projection_required,E_k_basis_lattice,PASS,True
5,projection_required,F_fixed_template_ssvd,PASS,True
6,projection_required,H_full_projection,PASS,True
7,model_sampling_required,J_one_step_casal_lite,FAIL,False
8,model_sampling_required,M_final_output_zT,PASS,True
9,model_sampling_required,N_csp_reconstruction_best_of_k,PASS,True
